# 外乱・不整地の追加実験
01/02と同じColabランタイムで実行する。02で作成した自分の指示歩行checkpointとconfig.jsonを使う。下の設定・対象実験を確認して必要なセルだけ実行する。全セル一括実行はしない。GPU計算とDriveへの保存が発生する。公開版Notebookの全工程再実行は未検証。


In [ ]:
from pathlib import Path
import os,json,subprocess,datetime,uuid,shutil
BASE=Path("/content/genesis-go2")
PY=BASE/"venv/bin/python"
SOURCE=BASE/"source"
ENV=dict(os.environ,MPLBACKEND="Agg",PYOPENGL_PLATFORM="egl",OMP_NUM_THREADS="2")
CHECKPOINT=Path("") # 02で作ったcheckpoint.ptの絶対パスに変更
assert CHECKPOINT.is_file(),"Specify your command checkpoint.pt"
assert (CHECKPOINT.parent/"config.json").is_file()


In [ ]:
FILES={'command-config.json': '{\n  "schema": 1,\n  "dt": 0.02,\n  "hold_s": 4.0,\n  "evaluation_joint_jitter_rad": 0.01,\n  "stages": {\n    "speed": {"weights": [0.2, 0.8, 0.0, 0.0], "vx_max": 0.6, "arc_vx_max": 0.6, "yaw_max": 0.0},\n    "full": {"weights": [0.2, 0.3, 0.2, 0.3], "vx_max": 0.8, "arc_vx_max": 0.6, "yaw_max": 0.6}\n  },\n  "thresholds": {\n    "vx_mae": 0.10, "vy_mae": 0.10, "yaw_mae": 0.15,\n    "turn_xy_speed": 0.10, "stop_xy_speed": 0.05, "stop_yaw_speed": 0.10,\n    "settle_s": 2.0, "dwell_s": 0.5, "exclude_s": 1.0,\n    "formal_success_rate": 0.9\n  },\n  "formal_seeds": [1000, 1001, 1002, 1003, 1004],\n  "formal_yaw_degrees": [-15, 0, 15],\n  "mirrors": [1, -1]\n}\n', 'command_pipeline.py': '"""Bounded staged training pipeline; inspect status.json without duplicating jobs."""\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport uuid\nimport zipfile\n\nfrom command_worker import backup, dump\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--session", type=Path, required=True)\n    parser.add_argument("--backup", type=Path, required=True)\n    parser.add_argument("--baseline", type=Path, required=True)\n    parser.add_argument("--num-envs", type=int, default=1024)\n    parser.add_argument("--iterations", type=int, default=301)\n    args = parser.parse_args()\n    args.session.mkdir(parents=True, exist_ok=False)\n    source = Path(__file__).parent\n    sequence = 0\n    history = []\n    chosen = None\n    def status(phase, **extra):\n        value = {"phase": phase, "updated_utc": datetime.now(timezone.utc).isoformat(),\n                 "history": history, **extra}\n        dump(args.session / "status.json", value)\n        args.backup.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(args.session / "status.json", args.backup / "status.json")\n        print(json.dumps({k: v for k, v in value.items() if k != "history"}), flush=True)\n\n    def job(mode, *extra, timeout=1800):\n        nonlocal sequence\n        sequence += 1\n        run = args.session / "runs" / f"{sequence:02d}-{mode}-{uuid.uuid4().hex[:6]}"\n        log = args.session / "logs" / (run.name + ".log")\n        log.parent.mkdir(parents=True, exist_ok=True)\n        command = [sys.executable, "-u", str(source / "command_worker.py"), mode,\n                   "--run", str(run), "--backup", str(args.backup / "runs"), *map(str, extra)]\n        status("running", mode=mode, run=str(run), command=command)\n        code = -1\n        try:\n            with log.open("w") as output:\n                process = subprocess.Popen(command, cwd=source, stdout=output, stderr=subprocess.STDOUT,\n                                           env=dict(os.environ, MPLBACKEND="Agg", PYOPENGL_PLATFORM="egl",\n                                                    OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2"))\n                try:\n                    code = process.wait(timeout=timeout)\n                except BaseException:\n                    process.terminate()\n                    try:\n                        process.wait(timeout=15)\n                    except subprocess.TimeoutExpired:\n                        process.kill()\n                        process.wait()\n                    code = process.returncode\n                    raise\n        finally:\n            run.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(log, run / "run.log")\n            dump(run / "command.json", command)\n            (run / "exit-code.txt").write_text(str(code))\n            backup(run, args.backup / "runs")\n        if code:\n            raise RuntimeError(f"{mode} failed exit={code}: {log}")\n        metrics = json.loads((run / "metrics.json").read_text())\n        history.append({"mode": mode, "run": str(run), "metrics": metrics})\n        status("job_completed", mode=mode, run=str(run), success_rate=metrics.get("success_rate"))\n        return run, metrics\n\n    try:\n        smoke, _ = job("smoke", "--stage", "full")\n        resumed, resumed_metrics = job("smoke", "--resume", smoke / "checkpoint.pt", "--stage", "full")\n        assert resumed_metrics["optimizer_restored"] and resumed_metrics["start_iteration"] == 2\n        status("resume_verified", run=str(resumed))\n        previous = None\n        for batch in range(3):\n            load = ["--resume", previous / "checkpoint.pt"] if previous else []\n            previous, _ = job("train", "--stage", "speed", "--num-envs", args.num_envs,\n                              "--iterations", args.iterations, *load)\n            _, metrics = job("eval", "--stage", "speed", "--checkpoint", previous / "checkpoint.pt")\n            if metrics["success_rate"] == 1.0:\n                break\n        # Keep the capability scope intact even if stage A needs more work: report its\n        # failures and give the full task a bounded attempt; never relabel it success.\n        full_previous = None\n        for batch in range(3):\n            load = ["--resume", full_previous / "checkpoint.pt"] if full_previous else ["--initialize-from", previous / "checkpoint.pt"]\n            full_previous, _ = job("train", "--stage", "full", "--num-envs", args.num_envs,\n                                   "--iterations", args.iterations, *load)\n            _, metrics = job("eval", "--stage", "full", "--checkpoint", full_previous / "checkpoint.pt")\n            if metrics["success_rate"] == 1.0:\n                break\n        if metrics["success_rate"] < 1.0:\n            # Plan\'s first reward adjustment: increase only yaw tracking weight.\n            status("yaw_reward_adjustment", old=0.2, new=0.5)\n            for batch in range(3):\n                load = (["--initialize-from", full_previous / "checkpoint.pt", "--yaw-reward", 0.5]\n                        if batch == 0 else ["--resume", full_previous / "checkpoint.pt"])\n                full_previous, _ = job("train", "--stage", "full", "--num-envs", args.num_envs,\n                                       "--iterations", args.iterations, *load)\n                _, metrics = job("eval", "--stage", "full", "--checkpoint", full_previous / "checkpoint.pt")\n                if metrics["success_rate"] == 1.0:\n                    break\n        chosen = full_previous / "checkpoint.pt"\n        formal_run, formal = job("eval", "--stage", "full", "--formal", "--checkpoint", chosen)\n        baseline_run, baseline = job("eval", "--stage", "full", "--formal", "--checkpoint", args.baseline)\n        recording, _ = job("record", "--stage", "full", "--checkpoint", chosen, timeout=1800)\n        summary = {"checkpoint": str(chosen), "checkpoint_drive": str(args.backup / "runs" / chosen.parent.name / chosen.name),\n                   "formal_run": str(formal_run), "formal_success_rate": formal["success_rate"],\n                   "baseline_success_rate": baseline["success_rate"], "recording": str(recording),\n                   "target_met": formal["success_rate"] >= 0.9,\n                   "scope": "speed, stop, yaw commands on flat ground; no disturbance or navigation"}\n        dump(args.session / "summary.json", summary)\n        shutil.copy2(args.session / "summary.json", args.backup / "summary.json")\n        archive = args.session / "go2-command-results.zip"\n        # Export final results and recipe, not every training checkpoint in every batch.\n        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:\n            z.write(args.session / "summary.json", "summary.json")\n            for directory in [chosen.parent, formal_run, baseline_run, recording]:\n                for path in directory.iterdir():\n                    if path.is_file() and (not path.name.startswith("model_") or path.name == "checkpoint.pt"):\n                        z.write(path, str(path.relative_to(args.session)))\n        shutil.copy2(archive, args.backup / archive.name)\n        digest = hashlib.sha256(archive.read_bytes()).hexdigest()\n        if digest != hashlib.sha256((args.backup / archive.name).read_bytes()).hexdigest():\n            raise RuntimeError("Drive archive hash mismatch")\n        status("complete", **summary, archive=str(args.backup / archive.name), archive_sha256=digest)\n    except BaseException as exc:\n        status("failed", error=f"{type(exc).__name__}: {exc}", checkpoint=str(chosen) if chosen else None)\n        raise\n\n\nif __name__ == "__main__":\n    main()\n', 'finish_push.py': '"""Wait for existing evaluation, record comparison, export scalars and preserve files."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,csv,hashlib,json,shutil,subprocess,sys,time,zipfile\nfrom pathlib import Path\np=argparse.ArgumentParser();p.add_argument(\'--session\',type=Path,required=True);p.add_argument(\'--backup\',type=Path,required=True);p.add_argument(\'--old\',type=Path,required=True);a=p.parse_args()\nsource=Path(__file__).parent\nout=a.session/\'final-artifacts\';out.mkdir(exist_ok=False)\ndef save(status,**extra):\n    (out/\'status.json\').write_text(json.dumps(dict(phase=status,**extra),indent=2))\ndef command(cmd,log):\n    with (out/log).open(\'w\') as f:\n        subprocess.run(list(map(str,cmd)),stdout=f,stderr=subprocess.STDOUT,check=True,timeout=1800)\ntry:\n    save(\'waiting_evaluation\')\n    deadline=time.monotonic()+1800\n    while True:\n        state=json.loads((a.session/\'status.json\').read_text())\n        if state[\'phase\']==\'failed\':raise RuntimeError(state.get(\'error\'))\n        if state[\'phase\']==\'evaluation_complete_recording_pending\':break\n        if time.monotonic()>deadline:raise TimeoutError(\'evaluation still running\')\n        time.sleep(10)\n    save(\'recording\')\n    for name,checkpoint in [(\'old\',a.old),(\'new\',Path(state[\'selected\'][\'checkpoint\']))]:\n        command([sys.executable,source/\'record_push.py\',\'--checkpoint\',checkpoint,\'--run\',out/name,\'--label\',name],\'record-\'+name+\'.log\')\n    command([\'ffmpeg\',\'-n\',\'-i\',out/\'old/push.mp4\',\'-i\',out/\'new/push.mp4\',\'-filter_complex\',\'hstack=inputs=2\',\'-c:v\',\'libx264\',\'-pix_fmt\',\'yuv420p\',out/\'comparison.mp4\'],\'compose.log\')\n    import av\n    with av.open(str(out/\'comparison.mp4\')) as v:\n        stream=v.streams.video[0];fps=float(stream.average_rate)\n        frames=sum(1 for _ in v.decode(video=0))\n    assert frames==600 and fps==25,(frames,fps)\n    assert json.loads((out/\'old/initial-states.json\').read_text())==json.loads((out/\'new/initial-states.json\').read_text())\n    save(\'exporting_curves\')\n    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator\n    import matplotlib.pyplot as plt\n    rows=[];fig,axes=plt.subplots(2,1,figsize=(10,6))\n    for job in state[\'history\']:\n        run=Path(job[\'run\']);events=list(run.glob(\'**/events.out.tfevents.*\'))\n        if not events:continue\n        cfg=json.loads((run/\'config.json\').read_text())\n        label=run.name+\' \'+cfg[\'push_stage\']\n        dest=out/\'raw\'/run.name;dest.mkdir(parents=True)\n        for event in events:\n            shutil.copy2(event,dest/event.name)\n            acc=EventAccumulator(str(event),size_guidance={\'scalars\':0});acc.Reload()\n            for tag in acc.Tags()[\'scalars\']:\n                vals=acc.Scalars(tag)\n                rows.extend(dict(run=run.name,stage=cfg[\'push_stage\'],tag=tag,step=x.step,wall_time=x.wall_time,value=x.value) for x in vals)\n                if tag in [\'Train/mean_reward\',\'Train/mean_episode_length\']:\n                    idx=0 if tag.endswith(\'mean_reward\') else 1\n                    axes[idx].plot([x.step for x in vals],[x.value for x in vals],label=label);axes[idx].set_ylabel(tag)\n        if (run/\'run.log\').exists():shutil.copy2(run/\'run.log\',dest/\'run.log\')\n    assert rows\n    with (out/\'scalars.csv\').open(\'w\') as f:\n        w=csv.DictWriter(f,fieldnames=rows[0]);w.writeheader();w.writerows(rows)\n    for axis in axes:axis.legend(fontsize=7);axis.grid(alpha=.3)\n    fig.tight_layout();fig.savefig(out/\'learning-curves.png\');plt.close(fig)\n    save(\'saving\')\n    # Existing experiment directories are never deleted. Verify every copied file.\n    hashes={}\n    for path in a.session.rglob(\'*\'):\n        if not path.is_file() or path.name.endswith(\'.partial\') or path.name in [\'pipeline.lock\',\'status.json\']:continue\n        relative=path.relative_to(a.session)\n        destination=a.backup/relative;destination.parent.mkdir(parents=True,exist_ok=True)\n        shutil.copy2(path,destination)\n        digest=hashlib.sha256(path.read_bytes()).hexdigest()\n        assert hashlib.sha256(destination.read_bytes()).hexdigest()==digest\n        hashes[str(relative)]=digest\n    (out/\'file-sha256.json\').write_text(json.dumps(hashes,indent=2))\n    summary=dict(selected=state[\'selected\'],video_frames=frames,video_fps=fps,scalar_rows=len(rows),verified_files=len(hashes))\n    for name in [\'old\',\'new\']:\n        for mode in [\'formal\',\'regression\']:\n            run=Path(state[name+\'_\'+mode]);m=json.loads((run/\'metrics.json\').read_text())\n            summary[name+\'_\'+mode]=dict(run=str(run),successes=sum(t[\'success\'] for t in m[\'trials\']),count=len(m[\'trials\']))\n    (out/\'summary.json\').write_text(json.dumps(summary,indent=2))\n    save(\'artifacts_ready_local_snapshot_pending\',**summary)\n    shutil.copytree(out,a.backup/\'final-artifacts\',dirs_exist_ok=True)\n    archive=a.session/\'push-results.zip\'\n    with zipfile.ZipFile(archive,\'x\',zipfile.ZIP_DEFLATED) as z:\n        for path in a.session.rglob(\'*\'):\n            if path.is_file() and path!=archive and not path.name.endswith(\'.partial\'):\n                z.write(path,path.relative_to(a.session))\n    shutil.copy2(archive,a.backup/archive.name)\n    assert hashlib.sha256(archive.read_bytes()).digest()==hashlib.sha256((a.backup/archive.name).read_bytes()).digest()\n    print(json.dumps(summary,indent=2))\nexcept BaseException as e:\n    save(\'failed\',error=f\'{type(e).__name__}: {e}\');raise\n', 'worker.py': '"""Bounded CUDA training/evaluation; embedded in the standalone Colab notebook."""\nimport argparse\nimport copy\nimport csv\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\n\n\ndef write_json(path, value):\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("mode", choices=["smoke", "train", "eval"])\n    parser.add_argument("--run", type=Path, required=True)\n    parser.add_argument("--backup", type=Path)\n    parser.add_argument("--checkpoint", type=Path)\n    parser.add_argument("--num-envs", type=int, default=1024)\n    parser.add_argument("--iterations", type=int, default=301)\n    parser.add_argument("--seed", type=int, default=1)\n    parser.add_argument("--video", action="store_true")\n    args = parser.parse_args()\n    if args.num_envs < 1 or args.iterations < 1:\n        parser.error("num-envs and iterations must be positive")\n    # Never clear or reuse a run directory.\n    args.run.mkdir(parents=True, exist_ok=False)\n    # Colab exports an inline backend unavailable inside the isolated venv.\n    # Set this before Genesis imports matplotlib transitively.\n    os.environ["MPLBACKEND"] = "Agg"\n    import numpy as np\n    import torch\n    import genesis as gs\n    from go2_record_env import Go2Env\n    from go2_train import get_cfgs, get_train_cfg\n    from rsl_rl.runners import OnPolicyRunner\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU required. Select a Colab GPU runtime.")\n    torch.set_num_threads(2)\n    gs.init(backend=gs.cuda, precision="32", logging_level="warning",\n            seed=args.seed, performance_mode=True)\n    if args.checkpoint:\n        args.checkpoint = args.checkpoint.resolve()\n        saved = json.loads((args.checkpoint.parent / "config.json").read_text())\n        current = json.loads((Path(__file__).parent / "upstream.lock.json").read_text())\n        if saved["upstream"] != current:\n            raise RuntimeError("Checkpoint and current upstream pins differ")\n        cfgs = saved["cfgs"]\n        train_cfg = saved["train_cfg"]\n    else:\n        cfgs = get_cfgs()\n        train_cfg = get_train_cfg(args.run.name)\n        train_cfg["save_interval"] = 25\n    spec = json.loads((Path(__file__).parent / "upstream.lock.json").read_text())\n    config = {"args": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()},\n              "cfgs": cfgs, "train_cfg": train_cfg, "upstream": spec,\n              "control_dt_s": 0.02, "physics_substeps": 2,\n              "success_rule": "500 steps without reset, vx MAE <0.2 m/s, x displacement >1m",\n              "evaluation_seeds": [1000, 1001, 1002]}\n    write_json(args.run / "config.json", config)\n    write_json(args.run / "dependencies.json", {\n        p: importlib.metadata.version(p) for p in\n        ["genesis-world", "rsl-rl-lib", "torch", "quadrants", "tensorboard"]})\n    for name in ["worker.py", "go2_env.py", "go2_record_env.py", "go2_train.py", "upstream.lock.json"]:\n        shutil.copy2(Path(__file__).parent / name, args.run / name)\n\n    def backup():\n        if args.backup:\n            destination = args.backup / args.run.name\n            destination.mkdir(parents=True, exist_ok=True)\n            for path in args.run.iterdir():\n                if path.is_file():\n                    temp = destination / (path.name + ".partial")\n                    shutil.copy2(path, temp)\n                    temp.replace(destination / path.name)\n\n    class SavedRunner(OnPolicyRunner):\n        def save(self, path, infos=None):\n            super().save(path, infos)\n            backup()\n\n    n = 8 if args.mode == "smoke" else (1 if args.mode == "eval" else args.num_envs)\n    env = Go2Env(n, *copy.deepcopy(cfgs), record=args.video)\n    obs = env.reset()\n    assert obs["policy"].shape == (n, 45)\n    runner = SavedRunner(env, copy.deepcopy(train_cfg), str(args.run), device=gs.device)\n    if args.checkpoint:\n        runner.load(str(args.checkpoint), map_location=str(gs.device))\n        # Upstream stores the index of the last completed iteration.\n        runner.current_learning_iteration += 1\n    started = time.monotonic()\n    if args.mode in ["smoke", "train"]:\n        policy = runner.alg.get_policy()\n        before = [p.detach().clone() for p in policy.parameters()]\n        iterations = 2 if args.mode == "smoke" else args.iterations\n        runner.learn(num_learning_iterations=iterations, init_at_random_ep_len=True)\n        delta = sum((a - b).abs().sum().item() for a, b in zip(policy.parameters(), before))\n        assert delta > 0 and np.isfinite(delta), "PPO did not update finite parameters"\n        runner.save(str(args.run / "checkpoint.pt"))\n        metrics = {"status": "ppo_updates_completed", "iterations_this_run": iterations,\n                   "transitions_this_run": n * train_cfg["num_steps_per_env"] * iterations,\n                   "parameter_l1_change": delta, "walking_success": "not_evaluated"}\n    else:\n        if not args.checkpoint:\n            raise ValueError("eval requires --checkpoint")\n        policy = runner.get_inference_policy(device=gs.device)\n        rows, trials = [], []\n        # Identical initial pose and command; seed changes alone do not establish generalization.\n        for control in ["standing_pd", "ppo"]:\n            for trial_seed in [1000, 1001, 1002]:\n                random.seed(trial_seed)\n                np.random.seed(trial_seed)\n                torch.manual_seed(trial_seed)\n                obs = env.reset()\n                # Upstream reset uses skip_forward=True. get_pos() can still expose\n                # the previous trial\'s cached pose until the first physics step.\n                origin = np.asarray(env.env_cfg["base_init_pos"], dtype=float).copy()\n                errors = []\n                fell = False\n                writer = None\n                # Streaming avoids keeping 250 RGB frames in RAM.\n                if args.video:\n                    import av\n                    video = args.run / f"{control}-{trial_seed}.mp4"\n                    writer = av.open(str(video), "w")\n                    stream = writer.add_stream("libx264", rate=25)\n                    stream.width, stream.height, stream.pix_fmt = 640, 480, "yuv420p"\n                try:\n                    # Environment buffers survive across trials; inference tensors cannot\n                    # be mutated by the next reset outside InferenceMode.\n                    with torch.no_grad():\n                        for step in range(500):\n                            actions = policy(obs) if control == "ppo" else torch.zeros((1, 12), device=gs.device)\n                            obs, rewards, dones, _ = env.step(actions)\n                            if not torch.isfinite(obs["policy"]).all() or not torch.isfinite(rewards).all():\n                                raise RuntimeError("Non-finite rollout")\n                            done = bool(dones[0].item())\n                            velocity = env.base_lin_vel[0].detach().cpu().tolist()\n                            pos = env.robot.get_pos()[0].detach().cpu().numpy()\n                            rows.append([control, trial_seed, step, (step + 1) * 0.02,\n                                         *velocity, *pos.tolist(), float(rewards[0]), done])\n                            # Exclude initial 1 second of settling; never count a reset state as tracking.\n                            if step >= 50 and not done:\n                                errors.append(abs(velocity[0] - 0.5))\n                            if writer and step % 2 == 0:\n                                env.camera.set_pose(pos=pos + np.array([1.7, -2.0, 1.2]),\n                                                    lookat=pos + np.array([0, 0, -0.1]))\n                                rgb = env.camera.render()[0]\n                                for packet in stream.encode(av.VideoFrame.from_ndarray(rgb, format="rgb24")):\n                                    writer.mux(packet)\n                            if done:\n                                fell = True\n                                break\n                finally:\n                    if writer:\n                        for packet in stream.encode():\n                            writer.mux(packet)\n                        writer.close()\n                mae = float(np.mean(errors)) if errors else None\n                displacement = float(pos[0] - origin[0]) if not fell else None\n                trials.append({"control": control, "seed": trial_seed, "steps": step + 1,\n                               "terminated_early": fell, "vx_mae_m_s": mae,\n                               "x_displacement_m": displacement,\n                               "success": not fell and mae is not None and mae < 0.2 and displacement > 1.0})\n        with (args.run / "rollout.csv").open("w") as output:\n            csvwriter = csv.writer(output)\n            csvwriter.writerow(["control", "seed", "step", "time_s", "vx", "vy", "vz",\n                                "x", "y", "z", "reward", "done"])\n            csvwriter.writerows(rows)\n        metrics = {"status": "evaluated", "trials": trials,\n                   "ppo_success_rate": sum(t["success"] for t in trials if t["control"] == "ppo") / 3,\n                   "scope": "fixed initial pose, fixed forward command; no turning or disturbance test"}\n    metrics["elapsed_s"] = time.monotonic() - started\n    write_json(args.run / "metrics.json", metrics)\n    backup()\n    print(json.dumps(metrics, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'upstream.lock.json': '{\n  "repository": "https://github.com/Genesis-Embodied-AI/Genesis",\n  "commit": "1ac23c39b32e567cc76e20a467fee99aaad257b7",\n  "engine_version": "1.4.5",\n  "files": {\n    "go2_env.py": "22a08b9f1f5c765732a7b3d04c98b3cb6a25b565aa024479a0526bbc654c1a63",\n    "go2_train.py": "9d78b33a90e8833be648c302a818f1b10ebfb632d0f5de5214f68bf1d714cb00"\n  }\n}\n', 'terrain_preflight.py': '"""Bounded sequential smoke, same-map resume and baseline dev. No training jobs."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,json,subprocess,sys,uuid,time,hashlib,shutil\nfrom pathlib import Path\nfrom terrain_maps import generate\nimport numpy as np\np=argparse.ArgumentParser();p.add_argument(\'--session\',type=Path,required=True);p.add_argument(\'--checkpoint\',type=Path,required=True);p.add_argument(\'--calibration\',type=Path,required=True);p.add_argument(\'--backup\',type=Path,required=True);a=p.parse_args()\nsource=Path(__file__).parent\nstate=dict(phase=\'preflight\',history=[],started=time.time())\ndef save():\n    temp=a.session/\'preflight-status.json.partial\';temp.write_text(json.dumps(state,indent=2));temp.replace(a.session/\'preflight-status.json\')\n    a.backup.mkdir(parents=True,exist_ok=True);shutil.copy2(a.session/\'preflight-status.json\',a.backup/\'preflight-status.json\')\ndef job(mode,checkpoint,seed,amplitude,extra=()):\n    run=a.session/\'jobs\'/(f\'{len(state["history"]):02d}-{mode}-\'+uuid.uuid4().hex[:6])\n    run.parent.mkdir(exist_ok=True)\n    cmd=[sys.executable,str(source/\'terrain_worker.py\'),mode,\'--run\',str(run),\'--checkpoint\',str(checkpoint),\'--map-seed\',str(seed),\'--amplitude\',str(amplitude),\'--calibration\',str(a.calibration),\'--backup\',str(a.backup/\'jobs\'),*extra]\n    log=run.with_suffix(\'.log\');state.update(current_run=str(run),command=cmd);save()\n    with log.open(\'w\') as f:\n        process=subprocess.Popen(cmd,stdout=f,stderr=subprocess.STDOUT,cwd=source)\n        state[\'child_pid\']=process.pid;save()\n        try:code=process.wait(timeout=1800)\n        except subprocess.TimeoutExpired:\n            process.terminate();process.wait(timeout=30);raise\n    if run.exists():shutil.copy2(log,run/\'run.log\');(run/\'exit-code.txt\').write_text(str(code))\n    assert code==0,f\'{mode} exit {code}; {log}\'\n    metrics=json.loads((run/\'metrics.json\').read_text())\n    state[\'history\'].append(dict(run=str(run),mode=mode,map_seed=seed,amplitude=amplitude,metrics=metrics));save()\n    return run\ntry:\n    assert hashlib.sha256(a.checkpoint.read_bytes()).hexdigest()==\'748a97c8a9eb1a1debb3d35e464518a850f731bb07764a821bd4ffe6c3435541\'\n    # Freeze formal maps before any gradient update, without evaluating them.\n    formal=a.session/\'formal-maps\';formal.mkdir(exist_ok=False);manifest=[]\n    for amplitude in [.01,.02,.04]:\n        for seed in range(200,205):\n            h,meta=generate(seed,amplitude);np.savez(formal/f\'{seed}-{amplitude}.npz\',height=h);manifest.append(meta)\n    (formal/\'manifest.json\').write_text(json.dumps(manifest,indent=2));shutil.copytree(formal,a.backup/\'formal-maps\')\n    state[\'phase\']=\'smoke\';save();r=job(\'smoke\',a.checkpoint,10,.01)\n    state[\'phase\']=\'resume_smoke\';save();job(\'smoke\',r/\'checkpoint.pt\',10,.01,[\'--resume\'])\n    state[\'phase\']=\'baseline_dev\';save()\n    for amplitude in [.01,.02,.04]:\n        for seed in range(100,103):job(\'eval\',a.checkpoint,seed,amplitude,[\'--initial-count\',\'2\'])\n    state[\'phase\']=\'preflight_complete_training_pending\';save()\nexcept BaseException as e:\n    state.update(phase=\'failed\',error=f\'{type(e).__name__}: {e}\');save();raise\n', 'calibrate_terrain_sampler.py': '"""Check the installed surface-height API against both triangles in two envs."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport genesis as gs\np=argparse.ArgumentParser();p.add_argument(\'--output\',type=Path,required=True);a=p.parse_args()\ngs.init(backend=gs.cuda,precision=\'32\',logging_level=\'warning\',seed=91)\nscene=gs.Scene(show_viewer=False)\nh=np.array([[0,10,0],[20,50,10],[0,10,0]],dtype=float)\nt=scene.add_entity(gs.morphs.Terrain(height_field=h,horizontal_scale=.1,vertical_scale=.001,pos=(-.1,-.1,0)))\nscene.build(n_envs=2)\nuv=np.array([[[.2,.3],[.7,.8]],[[.6,.1],[.9,.6]]])\nxy=torch.tensor(uv*.1-.1,device=gs.device,dtype=torch.float32)\nactual=t.get_terrain_height(xy).cpu().numpy()\nu,v=uv[...,0],uv[...,1]\nexpected=np.where(u+v<=1,20*u+10*v,50+(1-v)*(20-50)+(1-u)*(10-50))*.001\nerror=float(np.abs(actual-expected).max())\nassert np.isfinite(actual).all() and error<1e-6,(actual,expected,error)\nresult=dict(status=\'passed\',method=\'Genesis Terrain.get_terrain_height; piecewise planar\',max_error_m=error,actual=actual.tolist(),expected=expected.tolist())\na.output.write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))\n', 'terrain_retry_spec.json': '{\n  "version": 2,\n  "training_maps": [[11, 0.02], [12, 0.02], [13, 0.04]],\n  "iterations_per_map": 301,\n  "num_envs": 256,\n  "candidates": [\n    {"name": "low_lr", "learning_rate": 0.0001, "posture_scale": 0.0},\n    {"name": "low_lr_posture", "learning_rate": 0.0001, "posture_scale": -0.5}\n  ],\n  "dev_seeds": [100, 101, 102],\n  "formal_seeds": [400, 401, 402, 403, 404],\n  "amplitudes": [0.01, 0.02, 0.04],\n  "selection": "flat >=27/30; dev successes descending, strong descending, candidate order ties; fallback old",\n  "time_cap_seconds": 7200\n}\n', 'terrain_maps.py': '"""Deterministic metric height fields; no simulator dependency."""\nimport hashlib\nimport numpy as np\n\n\ndef generate(seed, amplitude, size=24., spacing=.1, coarse=.4):\n    if amplitude < 0 or spacing <= 0 or size < 4:\n        raise ValueError(\'invalid terrain dimensions\')\n    rng=np.random.default_rng(seed)\n    coarse_axis=np.linspace(-size/2,size/2,round(size/coarse)+1)\n    axis=np.linspace(-size/2,size/2,round(size/spacing)+1)\n    values=rng.uniform(-1,1,(len(coarse_axis),len(coarse_axis)))\n    intermediate=np.stack([np.interp(axis,coarse_axis,row) for row in values])\n    height=np.stack([np.interp(axis,coarse_axis,intermediate[:,j]) for j in range(len(axis))],axis=1)\n    height-=height.mean()\n    height/=max(float(np.abs(height).max()),1e-12)\n    x,y=np.meshgrid(axis,axis,indexing=\'ij\')\n    blend=np.clip((np.maximum(np.abs(x),np.abs(y))-1.)/.5,0,1)\n    height=np.rint(height*amplitude*blend/.001).astype(np.int16)\n    return height,dict(seed=int(seed),amplitude_m=amplitude,horizontal_scale=spacing,vertical_scale=.001,\n        pos=[-size/2,-size/2,0],shape=list(height.shape),sha256=hashlib.sha256(height.tobytes()).hexdigest(),\n        rms_m=float(np.sqrt(np.mean((height.astype(float)*.001)**2))),\n        max_abs_m=float(np.abs(height).max()*.001))\n\n\ndef sample(height,metadata,xy):\n    """Bilinear reference sampler. Physical interpolation must be calibrated separately."""\n    coords=(np.asarray(xy)-np.asarray(metadata[\'pos\'][:2]))/metadata[\'horizontal_scale\']\n    if np.any(coords<0) or np.any(coords>np.asarray(height.shape)-1):\n        raise ValueError(\'outside terrain\')\n    lower=np.minimum(np.floor(coords).astype(int),np.asarray(height.shape)-2)\n    f=coords-lower;i,j=lower[...,0],lower[...,1];u,v=f[...,0],f[...,1]\n    return ((1-u)*(1-v)*height[i,j]+u*(1-v)*height[i+1,j]+(1-u)*v*height[i,j+1]+u*v*height[i+1,j+1])*metadata[\'vertical_scale\']\n', 'pushes.py': '"""Pure functions for timed physical pushes and recovery scoring."""\nimport math\n\nDT = .02\nLEVELS = (.10, .20, .30)\n\n\ndef lateral_force(yaw, sign, alpha, mass):\n    if sign not in (-1, 1) or not (mass > 0 and math.isfinite(mass)) or not (0 <= alpha <= .30):\n        raise ValueError(\'Invalid push parameters\')\n    magnitude = sign * alpha * mass * 9.81\n    return [-math.sin(yaw)*magnitude, math.cos(yaw)*magnitude, 0.0]\n\n\ndef pulse_active(step, start=200, duration=10):\n    return start <= step < start + duration\n\n\ndef cases(formal=False):\n    seeds = range(2000, 2010) if formal else range(90, 95)\n    return [dict(alpha=a, sign=s, seed=seed) for a in LEVELS for s in (-1, 1) for seed in seeds]\n\n\ndef score(rows, force_end=4.2):\n    def good(r):\n        return not r[\'done\'] and abs(r[\'vx\']-.5) <= .1 and abs(r[\'vy\']) <= .1 and abs(r[\'wz\']) <= .15\n    before = [r for r in rows if 3.5 < r[\'time\'] <= 4.0]\n    streak, recovery = 0, None\n    for r in rows:\n        if r[\'time\'] <= force_end:\n            continue\n        streak = streak+1 if good(r) else 0\n        if streak >= 25 and recovery is None:\n            recovery = r[\'time\']-force_end\n    complete = len(rows) == 600 and not any(r[\'done\'] for r in rows)\n    return dict(success=bool(complete and len(before)==25 and all(map(good,before)) and recovery is not None and recovery <= 2),\n                recovery_s=recovery, completed=complete)\n', 'terrain_env.py': '"""Terrain policy wrapper. GPU parity and sampler probes required before training."""\nimport copy\nimport numpy as np\nimport torch\nfrom terrain_command_base import CommandEnv\n\nclass TerrainEnv(CommandEnv):\n    def __init__(self,num_envs,cfgs,terrain_file,*,evaluation=False,record=False):\n        cfgs=copy.deepcopy(cfgs);cfgs[0][\'_terrain_file\']=str(terrain_file)\n        # Allocated before super because reward functions may be bound during initialization.\n        self.height_cpu=np.load(terrain_file)[\'height\'].copy()\n        self.height_tensor=None\n        super().__init__(num_envs,cfgs,stage=\'full\',evaluation=evaluation,record=record)\n        self.height_tensor=torch.as_tensor(self.height_cpu,device=self.device,dtype=torch.float32)*.001\n    def ground_height(self,xy):\n        # Installed Genesis 1.4.5 exposes the actual piecewise-planar surface.\n        # A per-environment point dimension avoids querying every point in every env.\n        return self.terrain.get_terrain_height(xy[:,None,:]).reshape(-1)\n    def _reward_base_height(self):\n        return torch.square(self.base_pos[:,2]-self.ground_height(self.base_pos[:,:2])-self.reward_cfg[\'base_height_target\'])\n', 'push_pipeline.py': '"""Bounded asynchronous pipeline; all jobs create fresh immutable run directories."""\nimport argparse,json,os,shutil,subprocess,sys,time,uuid\nfrom pathlib import Path\nfrom command_worker import dump\nfrom audit_pushes import audit\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--session\',type=Path,required=True);p.add_argument(\'--checkpoint\',type=Path,required=True);p.add_argument(\'--backup\',type=Path,required=True);a=p.parse_args()\n    source=Path(__file__).parent\n    a.session.mkdir(parents=True,exist_ok=True)\n    lock=a.session/\'pipeline.lock\'\n    fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY);os.write(fd,str(os.getpid()).encode());os.close(fd)\n    state=dict(phase=\'starting\',pid=os.getpid(),history=[],started=time.time())\n    def save():\n        dump(a.session/\'status.json\',state)\n        a.backup.mkdir(parents=True,exist_ok=True)\n        shutil.copy2(a.session/\'status.json\',a.backup/\'status.json\')\n    def job(mode,checkpoint,stage=\'weak\',resume=False,formal=False,iterations=301):\n        name=f\'{len(state["history"]):02d}-{mode}-{uuid.uuid4().hex[:6]}\'\n        run=a.session/\'runs\'/name;run.parent.mkdir(exist_ok=True)\n        command=[sys.executable,\'-u\',str(source/\'push_worker.py\'),mode,\'--run\',str(run),\'--checkpoint\',str(checkpoint),\n                 \'--stage\',stage,\'--iterations\',str(iterations),\'--backup\',str(a.backup/\'runs\')]\n        if resume:command+=[\'--resume\']\n        if formal:command+=[\'--formal\']\n        state.update(phase=mode,current_run=str(run),command=command);save()\n        log=a.session/(name+\'.log\')\n        with log.open(\'w\') as f:\n            proc=subprocess.Popen(command,stdout=f,stderr=subprocess.STDOUT,env=dict(os.environ,MPLBACKEND=\'Agg\',OMP_NUM_THREADS=\'2\',OPENBLAS_NUM_THREADS=\'2\'))\n            state[\'child_pid\']=proc.pid;save()\n            try:code=proc.wait(timeout=1800)\n            except subprocess.TimeoutExpired:\n                proc.terminate()\n                try:proc.wait(timeout=15)\n                except subprocess.TimeoutExpired:proc.kill();proc.wait()\n                raise RuntimeError(\'bounded job timeout\')\n        run.mkdir(exist_ok=True)\n        shutil.copy2(log,run/\'run.log\');dump(run/\'command.json\',command);(run/\'exit-code.txt\').write_text(str(code))\n        shutil.copytree(run,a.backup/\'runs\'/name,dirs_exist_ok=True)\n        if code:raise RuntimeError(f\'{name} failed exit={code}; inspect log\')\n        metrics=json.loads((run/\'metrics.json\').read_text())\n        if mode==\'eval\':\n            result=audit(run);dump(run/\'audit.json\',result)\n            shutil.copy2(run/\'audit.json\',a.backup/\'runs\'/name/\'audit.json\')\n        state[\'history\'].append(dict(run=str(run),metrics=metrics));save()\n        return run,metrics\n    try:\n        assert json.loads((a.session/\'calibration/calibration.json\').read_text())[\'status\']==\'passed\'\n        assert json.loads((a.session/\'dev-baseline/metrics.json\').read_text())[\'status\']==\'evaluated\'\n        audit(a.session/\'dev-baseline\')\n        smoke,_=job(\'smoke\',a.checkpoint)\n        job(\'smoke\',smoke/\'checkpoint.pt\',resume=True)\n        checkpoint=a.checkpoint\n        candidates=[]\n        for stage,batches in [(\'weak\',2),(\'mixed\',3)]:\n            for batch in range(batches):\n                train,_=job(\'train\',checkpoint,stage,resume=batch>0)\n                checkpoint=train/\'checkpoint.pt\'\n                dev,metrics=job(\'eval\',checkpoint,stage)\n                regression,reg=job(\'regression\',checkpoint,stage)\n                candidates.append(dict(checkpoint=str(checkpoint),dev=metrics[\'success_rate\'],regression=reg[\'success_rate\']))\n                state[\'candidates\']=candidates;save()\n                if stage==\'weak\' and metrics[\'by_alpha\'][\'0.1\']>=.9:break\n                if stage==\'mixed\' and metrics[\'by_alpha\'][\'0.2\']>=.9 and reg[\'success_rate\']>=.9:break\n        eligible=[c for c in candidates if c[\'regression\']>=.9]\n        best=max(eligible or candidates,key=lambda c:(c[\'dev\'],c[\'regression\']))\n        state[\'selected\']=best;save()\n        for name,checkpoint in [(\'old\',a.checkpoint),(\'new\',Path(best[\'checkpoint\']))]:\n            run,_=job(\'eval\',checkpoint,formal=True)\n            state[name+\'_formal\']=str(run)\n            run,_=job(\'regression\',checkpoint,formal=True)\n            state[name+\'_regression\']=str(run)\n        state[\'phase\']=\'evaluation_complete_recording_pending\';save()\n    except BaseException as e:\n        state.update(phase=\'failed\',error=f\'{type(e).__name__}: {e}\');save();raise\n\nif __name__==\'__main__\':main()\n', 'push-config.json': '{\n  "version": 1,\n  "control_dt_s": 0.02,\n  "physics_substeps": 2,\n  "force": "alpha * robot_mass_kg * 9.81",\n  "reference": "link_COM",\n  "world_vector_fixed_at_onset": true,\n  "pulse_steps": 10,\n  "levels": [0.1, 0.2, 0.3],\n  "dev_seeds": [90, 91, 92, 93, 94],\n  "formal_seeds": [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009],\n  "formal_start_step": 200,\n  "evaluation_steps": 600,\n  "precondition_steps": 25,\n  "recovery_dwell_steps": 25,\n  "recovery_deadline_s": 2.0,\n  "velocity_tolerance": [0.1, 0.1, 0.15],\n  "unforced_episode_probability": 0.5,\n  "weak_range": [0.05, 0.15],\n  "mixed_range": [0.05, 0.3]\n}\n', 'push_env.py': '"""Physical lateral push wrapper; no additional simulation steps or observations."""\nimport torch\nimport genesis as gs\nfrom genesis.utils.geom import quat_to_xyz\nfrom command_env import CommandEnv\n\n\nclass PushEnv(CommandEnv):\n    def __init__(self, num_envs, cfgs, *, training_range=None, evaluation=False, record=False):\n        self.push_ready = False\n        self.training_range = training_range\n        super().__init__(num_envs, cfgs, stage=\'full\', evaluation=evaluation, record=record)\n        self.trunk = self.robot.get_link(\'base\')\n        self.mass = float(self.robot.get_mass().flatten()[0])\n        assert 1 < self.mass < 100\n        self.push_clock = torch.zeros(num_envs,device=self.device,dtype=torch.long)\n        self.next_push = torch.zeros_like(self.push_clock)\n        self.remaining = torch.zeros_like(self.push_clock)\n        self.enabled = torch.zeros(num_envs,device=self.device,dtype=torch.bool)\n        self.force = torch.zeros((num_envs,3),device=self.device)\n        self.applied_force = self.force.clone()\n        self.pulses_applied = 0\n        self.push_ready = True\n        self._reset_push(None)\n\n    def _reset_push(self, mask):\n        mask = torch.ones(self.num_envs,device=self.device,dtype=torch.bool) if mask is None else mask\n        self.push_clock[mask] = 0\n        self.remaining[mask] = 0\n        self.force[mask] = 0\n        self.enabled[mask] = torch.rand(int(mask.sum()),device=self.device)<.5\n        self.next_push[mask] = torch.randint(150,301,(int(mask.sum()),),device=self.device)\n\n    def _reset_idx(self, envs_idx=None):\n        super()._reset_idx(envs_idx)\n        if self.push_ready:\n            self._reset_push(envs_idx)\n\n    def start_push(self, mask, alpha, sign):\n        yaw = torch.deg2rad(quat_to_xyz(self.robot.get_quat(),rpy=True,degrees=True)[:,2])\n        magnitude = torch.as_tensor(alpha,device=self.device)*self.mass*9.81*torch.as_tensor(sign,device=self.device)\n        vector = torch.stack((-torch.sin(yaw)*magnitude,torch.cos(yaw)*magnitude,torch.zeros_like(yaw)),dim=1)\n        self.force[mask] = vector[mask]\n        self.remaining[mask] = 10\n        self.pulses_applied += int(mask.sum())\n\n    def step(self, actions):\n        if self.training_range is not None:\n            onset = self.enabled & (self.push_clock == self.next_push)\n            low, high = self.training_range\n            alpha = low+(high-low)*torch.rand(self.num_envs,device=self.device)\n            sign = torch.where(torch.rand(self.num_envs,device=self.device)<.5,-1.,1.)\n            self.start_push(onset,alpha,sign)\n            self.next_push[onset] += torch.randint(200,351,(int(onset.sum()),),device=self.device)\n        self.applied_force = torch.where((self.remaining>0)[:,None],self.force,0.).clone()\n        if bool((self.remaining>0).any()):\n            self.scene.rigid_solver.apply_links_external_wrench(\n                force=self.applied_force[:,None,:], links_idx=[self.trunk.idx],\n                ref=gs.link_ref_frame.link_COM,local=False)\n        # Advance independent clock before super: reset hook must leave reset envs at zero.\n        self.push_clock += 1\n        self.remaining.clamp_(min=0)\n        self.remaining.sub_(1).clamp_(min=0)\n        return super().step(actions)\n', 'finalize.py': '"""Audit recorded trajectories, decode videos, and archive a completed Colab run."""\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport sys\nimport zipfile\n\nimport av\nfrom PIL import Image, ImageDraw\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef main():\n    evaluation, training, backup = map(Path, sys.argv[1:4])\n    metrics_path = evaluation / "metrics.json"\n    config = json.loads((evaluation / "config.json").read_text())\n    initial_x = float(config["cfgs"][0]["base_init_pos"][0])\n    original = evaluation / "metrics-before-origin-fix.json"\n    if not original.exists():\n        shutil.copy2(metrics_path, original)\n    metrics = json.loads(original.read_text())\n    with (evaluation / "rollout.csv").open() as stream:\n        rows = list(csv.DictReader(stream))\n    for trial in metrics["trials"]:\n        group = [r for r in rows if r["control"] == trial["control"] and int(r["seed"]) == trial["seed"]]\n        assert group and abs(float(group[0]["x"]) - initial_x) < 0.2\n        early = any(r["done"].lower() == "true" for r in group)\n        errors = [abs(float(r["vx"]) - 0.5) for r in group if int(r["step"]) >= 50 and r["done"].lower() != "true"]\n        mae = sum(errors) / len(errors) if errors else None\n        displacement = float(group[-1]["x"]) - initial_x if not early else None\n        trial.update(steps=len(group), terminated_early=early, vx_mae_m_s=mae,\n                     x_displacement_m=displacement,\n                     success=len(group) == 500 and not early and mae is not None and mae < 0.2 and displacement > 1.0)\n    metrics["ppo_success_rate"] = sum(t["success"] for t in metrics["trials"] if t["control"] == "ppo") / 3\n    metrics["measurement_correction"] = {\n        "reason": "reset uses skip_forward=True; pre-step get_pos may return the preceding trial\'s cached pose",\n        "method": "recompute from logged world x at final step minus configured reset x; no physics/training rerun",\n        "original_preserved": original.name,\n    }\n    video_checks = []\n    canvas = Image.new("RGB", (960, 520), "white")\n    for video in sorted(evaluation.glob("*.mp4")):\n        count = 0\n        samples = {}\n        with av.open(str(video)) as container:\n            fps = float(container.streams.video[0].average_rate)\n            for count, frame in enumerate(container.decode(video=0), start=1):\n                if count in [26, 126, 226]:\n                    samples[count] = frame.to_image()\n        assert count == 250 and fps == 25, (video.name, count, fps)\n        video_checks.append({"file": video.name, "frames": count, "fps": fps,\n                             "bytes": video.stat().st_size, "sha256": sha256(video)})\n        if video.name in ["standing_pd-1000.mp4", "ppo-1000.mp4"]:\n            y = 0 if video.name.startswith("standing") else 260\n            for column, (number, image) in enumerate(samples.items()):\n                canvas.paste(image.resize((320, 240)), (column * 320, y + 20))\n                ImageDraw.Draw(canvas).text((column * 320 + 5, y + 4),\n                                           f"{video.stem} t={(number - 1) / 25:.0f}s", fill="black")\n    canvas.save(evaluation / "walking-preview.jpg")\n    metrics["videos"] = video_checks\n    metrics_path.write_text(json.dumps(metrics, indent=2) + "\\n")\n    shutil.copy2(Path(__file__), evaluation / "finalize.py")\n    shutil.copytree(evaluation, backup / evaluation.name, dirs_exist_ok=True)\n    checkpoint = training / "checkpoint.pt"\n    assert sha256(checkpoint) == sha256(backup / training.name / "checkpoint.pt")\n    archive = evaluation.parent.parent / "genesis-go2-results.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:\n        for directory in [training, evaluation]:\n            for path in directory.iterdir():\n                if path.is_file():\n                    z.write(path, f"{directory.name}/{path.name}")\n    saved_archive = backup.parent / archive.name\n    shutil.copy2(archive, saved_archive)\n    assert sha256(archive) == sha256(saved_archive)\n    print(json.dumps({"metrics": metrics, "archive": str(saved_archive),\n                      "archive_bytes": archive.stat().st_size, "archive_sha256": sha256(archive),\n                      "checkpoint_drive_sha256": sha256(checkpoint)}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'command_worker.py': '"""CUDA entry point for command-conditioned Go2 training, evaluation and teleoperation."""\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\n\nfrom commands import CONFIG, cases, command_at, score_trial\n\n\ndef dump(path, value):\n    temp = path.with_suffix(path.suffix + ".partial")\n    temp.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n    temp.replace(path)\n\n\ndef backup(run, destination):\n    if destination:\n        target = destination / run.name\n        target.mkdir(parents=True, exist_ok=True)\n        for path in run.iterdir():\n            if path.is_file() and not path.name.endswith(".partial"):\n                temp = target / (path.name + ".partial")\n                shutil.copy2(path, temp)\n                temp.replace(target / path.name)\n\n\ndef evaluate(env, runner, args, evaluation_stage):\n    import torch\n    import numpy as np\n    from PIL import Image, ImageDraw\n    trial_cases = cases(args.formal, evaluation_stage, args.mode in ["record", "teleop"])\n    obs = env.reset_with_yaws([c["yaw_degrees"] for c in trial_cases], [c["seed"] for c in trial_cases])\n    dump(args.run / "initial-states.json", [dict(c, qpos=q) for c, q in zip(trial_cases, env.evaluation_initial_qpos)])\n    policy = runner.get_inference_policy(device=env.device)\n    rows = [[] for _ in trial_cases]\n    active = [True] * len(trial_cases)\n    video_writer, video_stream = None, None\n    if args.mode in ["record", "teleop"]:\n        import av\n        video_writer = av.open(str(args.run / "commands.mp4"), "w")\n        video_stream = video_writer.add_stream("libx264", rate=25)\n        video_stream.width, video_stream.height, video_stream.pix_fmt = 640, 480, "yuv420p"\n    started = time.monotonic()\n    try:\n        with torch.no_grad():\n            for step in range(1500):\n                applied, segment_ids = [], []\n                for c in trial_cases:\n                    segment, cmd = command_at(step, evaluation_stage, c["mirror"])\n                    if args.mode == "teleop":\n                        requested = json.loads(args.control.read_text())\n                        cmd = [max(0, min(0.8, float(requested["vx"]))), 0,\n                               max(-0.6, min(0.6, float(requested["yaw"])))]\n                        if not all(math.isfinite(v) for v in cmd):\n                            raise ValueError("Non-finite control")\n                    applied.append(list(cmd))\n                    segment_ids.append(segment)\n                obs = env.set_commands(applied)\n                assert torch.allclose(obs["policy"][:, 6:9], env.commands * env.commands_scale)\n                actions = policy(obs)\n                if not torch.isfinite(actions).all():\n                    raise RuntimeError("Non-finite policy action")\n                obs, reward, _, _ = env.step(actions)\n                state = {k: v.detach().cpu().numpy() for k, v in env.terminal.items()}\n                if not all(np.isfinite(v).all() for v in state.values()) or not torch.isfinite(obs["policy"]).all():\n                    raise RuntimeError("Non-finite physics/observation")\n                action_values = actions.detach().cpu().tolist()\n                reward_terms = {k: v.cpu().tolist() for k, v in env.reward_values.items()}\n                for i, c in enumerate(trial_cases):\n                    if not active[i]:\n                        continue\n                    pos = state["base_pos"][i]\n                    quat = state["base_quat"][i]\n                    vel = state["base_lin_vel"][i]\n                    angular = state["base_ang_vel"][i]\n                    euler = state["base_euler"][i]\n                    done = bool(state["reset_buf"][i])\n                    row = {"step": step, "time": (step + 1) * 0.02, "segment": segment_ids[i],\n                           "cmd_vx": applied[i][0], "cmd_vy": applied[i][1], "cmd_yaw": applied[i][2],\n                           "vx": float(vel[0]), "vy": float(vel[1]), "vz": float(vel[2]),\n                           "wz": float(angular[2]), "x": float(pos[0]), "y": float(pos[1]), "z": float(pos[2]),\n                           "roll": float(euler[0]), "pitch": float(euler[1]),\n                           "yaw_world": math.atan2(2*(quat[0]*quat[3]+quat[1]*quat[2]), 1-2*(quat[2]**2+quat[3]**2)),\n                           "joint_speed_rms": float(np.sqrt(np.mean(state["dof_vel"][i] ** 2))),\n                           "reward": float(state["rew_buf"][i]), "done": done,\n                           "termination": "solver_error" if state["solver_error"][i] else ("roll_pitch_limit" if done else "none")}\n                    row.update({f"action_{j}": a for j, a in enumerate(action_values[i])})\n                    row.update({f"reward_{k}": v[i] for k, v in reward_terms.items()})\n                    rows[i].append(row)\n                    if done:\n                        active[i] = False\n                if video_writer and step % 2 == 0:\n                    row = rows[0][-1]\n                    if not row["done"]:\n                        pos = state["base_pos"][0]\n                        env.camera.set_pose(pos=pos + np.array([1.8, -2.0, 1.3]), lookat=pos)\n                        rgb = env.camera.render()[0]\n                        picture = Image.fromarray(rgb)\n                        draw = ImageDraw.Draw(picture)\n                        draw.rectangle((0, 0, 640, 56), fill="black")\n                        draw.text((8, 5), f\'t={row["time"]:.2f}s  command vx={row["cmd_vx"]:.2f}m/s yaw={row["cmd_yaw"]:.2f}rad/s\', fill="white")\n                        draw.text((8, 24), f\'actual vx={row["vx"]:.2f}m/s yaw={row["wz"]:.2f}rad/s  playback 1x\', fill="white")\n                        for packet in video_stream.encode(av.VideoFrame.from_ndarray(np.array(picture), format="rgb24")):\n                            video_writer.mux(packet)\n                        if args.mode == "teleop" and step % 10 == 0:\n                            picture.save(args.run / "latest.partial.jpg")\n                            (args.run / "latest.partial.jpg").replace(args.run / "latest.jpg")\n                if not any(active):\n                    break\n                if args.mode == "teleop":\n                    time.sleep(max(0, started + (step+1)*0.02 - time.monotonic()))\n    finally:\n        if video_writer:\n            for packet in video_stream.encode():\n                video_writer.mux(packet)\n            video_writer.close()\n        # Keep partial trajectories even if a renderer/physics failure interrupts the run.\n        with (args.run / "rollout.csv").open("w", newline="") as output:\n            nonempty = next((r for r in rows if r), None)\n            if nonempty:\n                writer = csv.DictWriter(output, fieldnames=["trial", "seed", "initial_yaw", "mirror", *nonempty[0]])\n                writer.writeheader()\n                for i, group in enumerate(rows):\n                    for r in group:\n                        writer.writerow({"trial": i, "seed": trial_cases[i]["seed"],\n                                         "initial_yaw": trial_cases[i]["yaw_degrees"], "mirror": trial_cases[i]["mirror"], **r})\n    trials = [{**c, **score_trial(r, evaluation_stage, c["mirror"])} for c, r in zip(trial_cases, rows)]\n    if args.mode == "teleop":\n        metrics = {"status": "manual_session_completed", "steps": len(rows[0]), "terminated_early": not active[0],\n                   "success_rate": None, "scope": "manual commands; not a formal evaluation"}\n    else:\n        metrics = {"status": "evaluated", "trials": trials,\n                   "success_rate": sum(t["success"] for t in trials) / len(trials),\n                   "formal": args.formal, "evaluation_stage": evaluation_stage,\n                   "scope": "flat ground; initial yaw and mirrored commands varied; no disturbance test"}\n        import matplotlib.pyplot as plt\n        example = rows[0]\n        fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)\n        for axis, measured, commanded, label in [(axes[0], "vx", "cmd_vx", "vx (m/s)"), (axes[1], "wz", "cmd_yaw", "yaw rate (rad/s)")]:\n            axis.plot([r["time"] for r in example], [r[commanded] for r in example], label="command")\n            axis.plot([r["time"] for r in example], [r[measured] for r in example], label="actual")\n            axis.set_ylabel(label)\n            axis.legend()\n        axes[1].set_xlabel("simulation time (s)")\n        fig.tight_layout()\n        fig.savefig(args.run / "tracking.png")\n        plt.close(fig)\n    return metrics\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("mode", choices=["smoke", "train", "eval", "record", "teleop"])\n    parser.add_argument("--run", required=True, type=Path)\n    parser.add_argument("--backup", type=Path)\n    parser.add_argument("--stage", choices=["speed", "full"])\n    parser.add_argument("--yaw-reward", type=float)\n    parser.add_argument("--num-envs", type=int, default=1024)\n    parser.add_argument("--iterations", type=int, default=301)\n    parser.add_argument("--seed", type=int, default=1)\n    parser.add_argument("--formal", action="store_true")\n    parser.add_argument("--control", type=Path)\n    group = parser.add_mutually_exclusive_group()\n    group.add_argument("--resume", type=Path)\n    group.add_argument("--initialize-from", type=Path)\n    group.add_argument("--checkpoint", type=Path)\n    args = parser.parse_args()\n    if args.num_envs < 1 or args.iterations < 1 or (args.mode == "teleop" and args.control is None):\n        parser.error("Invalid resource/control arguments")\n    if args.mode in ["eval", "record", "teleop"] and not args.checkpoint:\n        parser.error("Evaluation/recording requires --checkpoint")\n    args.run.mkdir(parents=True, exist_ok=False)\n    os.environ["MPLBACKEND"] = "Agg"\n    import torch\n    import genesis as gs\n    from command_env import CommandEnv\n    from go2_train import get_cfgs, get_train_cfg\n    from rsl_rl.runners import OnPolicyRunner\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA required")\n    torch.set_num_threads(2)\n    spec = json.loads(Path(__file__).with_name("upstream.lock.json").read_text())\n    checkpoint = args.resume or args.initialize_from or args.checkpoint\n    saved = None\n    if checkpoint:\n        saved = json.loads((checkpoint.parent / "config.json").read_text())\n        if saved["upstream"] != spec:\n            raise ValueError("Upstream pins differ")\n    stage = args.stage or (saved.get("command_stage", "speed") if saved else "speed")\n    cfgs = get_cfgs()\n    train_cfg = get_train_cfg(args.run.name)\n    train_cfg["save_interval"] = 25\n    if args.resume:\n        if stage != saved["command_stage"] or saved["command_config"] != CONFIG:\n            raise ValueError("Resume must preserve stage and command configuration; use initialize-from for a new task")\n        cfgs, train_cfg = saved["cfgs"], saved["train_cfg"]\n        if args.yaw_reward is not None and args.yaw_reward != cfgs[2]["reward_scales"]["tracking_ang_vel"]:\n            raise ValueError("Resume cannot change the reward")\n    elif saved and args.checkpoint:\n        cfgs, train_cfg = saved["cfgs"], saved["train_cfg"]\n    elif args.yaw_reward is not None:\n        cfgs[2]["reward_scales"]["tracking_ang_vel"] = args.yaw_reward\n    if args.mode in ["eval", "record", "teleop"]:\n        cfgs = copy.deepcopy(cfgs)\n        cfgs[0]["episode_length_s"] = 35.0\n    cfgs[0]["resampling_time_s"] = CONFIG["hold_s"]\n    config = {"task": "go2-commands-v1", "command_stage": stage, "command_config": CONFIG,\n              "cfgs": cfgs, "train_cfg": train_cfg, "upstream": spec, "seed": args.seed,\n              "args": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}}\n    dump(args.run / "config.json", config)\n    sources = {}\n    for name in ["command_worker.py", "commands.py", "command_env.py", "command-config.json", "go2_env.py", "go2_record_env.py", "go2_train.py", "upstream.lock.json"]:\n        source = Path(__file__).with_name(name)\n        shutil.copy2(source, args.run / name)\n        sources[name] = hashlib.sha256(source.read_bytes()).hexdigest()\n    dump(args.run / "source-sha256.json", sources)\n    dump(args.run / "dependencies.json", {p: importlib.metadata.version(p) for p in ["genesis-world", "torch", "rsl-rl-lib", "quadrants"]})\n    (args.run / "requirements.freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))\n    (args.run / "gpu.txt").write_text(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True))\n    gs.init(backend=gs.cuda, precision="32", logging_level="warning", seed=args.seed, performance_mode=True)\n    is_eval = args.mode in ["eval", "record", "teleop"]\n    n = len(cases(args.formal, stage, args.mode in ["record", "teleop"])) if is_eval else (8 if args.mode == "smoke" else args.num_envs)\n    env = CommandEnv(n, cfgs, stage=stage, evaluation=is_eval, record=args.mode in ["record", "teleop"])\n    class SavedRunner(OnPolicyRunner):\n        def save(self, path, infos=None):\n            super().save(path, infos)\n            backup(args.run, args.backup)\n    runner = SavedRunner(env, copy.deepcopy(train_cfg), str(args.run), device=gs.device)\n    if checkpoint:\n        load_cfg = {"actor": True, "critic": True, "optimizer": False, "iteration": False} if args.initialize_from else None\n        runner.load(str(checkpoint), load_cfg=load_cfg, map_location=str(gs.device))\n        if args.resume:\n            runner.current_learning_iteration += 1\n    before_iteration = runner.current_learning_iteration\n    started = time.monotonic()\n    if is_eval:\n        metrics = evaluate(env, runner, args, stage)\n    else:\n        assert env.get_observations()["policy"].shape == (n, 45)\n        # Verify command injection and exact zero stop observations before training.\n        env.set_commands(torch.zeros(n, 3, device=gs.device))\n        assert torch.equal(env.get_observations()["policy"][:, 6:9], torch.zeros(n, 3, device=gs.device))\n        env.reset()\n        before = [p.detach().clone() for p in runner.alg.get_policy().parameters()]\n        iterations = 2 if args.mode == "smoke" else args.iterations\n        runner.learn(num_learning_iterations=iterations, init_at_random_ep_len=True)\n        delta = sum((a-b).abs().sum().item() for a, b in zip(runner.alg.get_policy().parameters(), before))\n        if not math.isfinite(delta) or delta <= 0:\n            raise RuntimeError("No finite PPO update")\n        runner.save(str(args.run / "checkpoint.pt"))\n        metrics = {"status": "ppo_updates_completed", "iterations_this_run": iterations,\n                   "transitions_this_run": n * train_cfg["num_steps_per_env"] * iterations,\n                   "start_iteration": before_iteration, "last_completed_iteration": runner.current_learning_iteration,\n                   "optimizer_restored": bool(args.resume), "parameter_l1_change": delta}\n    metrics["elapsed_s"] = time.monotonic() - started\n    dump(args.run / "metrics.json", metrics)\n    backup(args.run, args.backup)\n    print(json.dumps(metrics, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except BaseException as exc:\n        # Parent orchestrator also persists an exit code when failure precedes imports.\n        print(f"COMMAND WORKER FAILED: {type(exc).__name__}: {exc}", flush=True)\n        raise\n', 'probe_terrain_policy.py': '"""Save policy trajectories for plane/zero terrain parity and weak-terrain contact."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,copy,json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport genesis as gs\np=argparse.ArgumentParser();p.add_argument(\'--source\',type=Path,required=True);p.add_argument(\'--checkpoint\',type=Path,required=True);p.add_argument(\'--run\',type=Path,required=True);p.add_argument(\'--kind\',choices=[\'plane\',\'zero\',\'weak\'],required=True);a=p.parse_args()\na.run.mkdir(parents=True,exist_ok=False)\nfrom rsl_rl.runners import OnPolicyRunner\nsaved=json.loads((a.checkpoint.parent/\'config.json\').read_text());cfgs=copy.deepcopy(saved[\'cfgs\'])\ngs.init(backend=gs.cuda,precision=\'32\',logging_level=\'warning\',seed=91,performance_mode=True)\nif a.kind==\'plane\':\n    from command_env import CommandEnv\n    env=CommandEnv(2,cfgs,evaluation=True)\nelse:\n    from terrain_maps import generate\n    from terrain_env import TerrainEnv\n    height,meta=generate(100,.01 if a.kind==\'weak\' else 0)\n    np.savez(a.run/\'terrain.npz\',height=height)\n    (a.run/\'terrain.json\').write_text(json.dumps(meta,indent=2))\n    env=TerrainEnv(2,cfgs,a.run/\'terrain.npz\',evaluation=True)\nrunner=OnPolicyRunner(env,copy.deepcopy(saved[\'train_cfg\']),None,device=gs.device)\nrunner.load(str(a.checkpoint),load_cfg=dict(actor=True,critic=True,optimizer=False,iteration=False),map_location=str(gs.device))\npolicy=runner.get_inference_policy(device=gs.device)\nenv.reset_with_yaws([0,90],[3000,3001]);rows=[]\nwith torch.no_grad():\n    for step in range(400 if a.kind==\'weak\' else 100):\n        obs=env.set_commands([[.5,0,0]]*2);action=policy(obs);env.step(action)\n        state=torch.cat([env.terminal[k].reshape(2,-1) for k in [\'base_pos\',\'base_quat\',\'base_lin_vel\',\'base_ang_vel\',\'dof_vel\']],dim=1)\n        assert torch.isfinite(state).all() and torch.isfinite(action).all()\n        rows.append(state.cpu().numpy());\n        if bool(env.terminal[\'reset_buf\'].any()):raise RuntimeError(f\'early termination at {step}\')\nnp.save(a.run/\'trajectory.npy\',np.stack(rows))\n(a.run/\'metrics.json\').write_text(json.dumps(dict(status=\'trajectory_completed\',steps=len(rows),kind=a.kind),indent=2))\nprint(\'Trajectory completed\',a.kind,len(rows))\n', 'finish_terrain.py': '"""Wait for evaluation, record, export raw/scalar curves, and verify Drive files."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,csv,hashlib,json,shutil,subprocess,sys,time,zipfile\nfrom pathlib import Path\nfrom audit_terrain import audit\np=argparse.ArgumentParser();p.add_argument(\'--session\',type=Path,required=True);p.add_argument(\'--backup\',type=Path,required=True);p.add_argument(\'--old\',type=Path,required=True);a=p.parse_args()\nsource=Path(__file__).parent;out=a.session/\'final-artifacts\';out.mkdir(exist_ok=False)\ndef status(phase,**extra):\n    (out/\'status.json\').write_text(json.dumps(dict(phase=phase,**extra),indent=2))\ndef command(cmd,log):\n    with (out/log).open(\'w\') as f:subprocess.run(list(map(str,cmd)),stdout=f,stderr=subprocess.STDOUT,check=True,timeout=1800)\ntry:\n    status(\'waiting_evaluation\');deadline=time.monotonic()+5400\n    while True:\n        state=json.loads((a.session/\'pipeline-status.json\').read_text())\n        if state[\'phase\']==\'failed\':raise RuntimeError(state.get(\'error\'))\n        if state[\'phase\']==\'terrain_evaluation_complete_flat_record_save_pending\':break\n        if time.monotonic()>deadline:raise TimeoutError(\'evaluation wait exceeded\')\n        time.sleep(10)\n    summary=dict(candidate=state[\'candidate\'],training_job_cap=state[\'training_job_cap\'],formal={},flat={})\n    for label in [\'old\',\'new\']:\n        items=[j for j in state[\'history\'] if j.get(\'formal_label\')==label]\n        if label==\'new\' and state.get(\'identical_policy_results_reused\'):items=[j for j in state[\'history\'] if j.get(\'formal_label\')==\'old\']\n        assert len(items)==15,(label,len(items))\n        by_amplitude={}\n        for amplitude in [.01,.02,.04]:\n            subset=[j for j in items if j[\'amplitude\']==amplitude];assert len(subset)==5\n            for j in subset:audit(j[\'run\'])\n            trials=[t for j in subset for t in j[\'metrics\'][\'trials\']];assert len(trials)==20\n            by_amplitude[str(amplitude)]=dict(successes=sum(t[\'success\'] for t in trials),count=len(trials))\n        summary[\'formal\'][label]=by_amplitude\n        flat=json.loads((Path(state[label+\'_regression\'])/\'metrics.json\').read_text());assert len(flat[\'trials\'])==30\n        summary[\'flat\'][label]=dict(successes=sum(t[\'success\'] for t in flat[\'trials\']),count=30)\n    old_items=[j for j in state[\'history\'] if j.get(\'formal_label\')==\'old\'];new_items=[j for j in state[\'history\'] if j.get(\'formal_label\')==\'new\']\n    if new_items:\n        for old in old_items:\n            new=next(j for j in new_items if j[\'seed\']==old[\'seed\'] and j[\'amplitude\']==old[\'amplitude\'])\n            assert (Path(old[\'run\'])/\'initial-states.json\').read_bytes()==(Path(new[\'run\'])/\'initial-states.json\').read_bytes()\n    status(\'recording\',**summary)\n    for label,checkpoint in [(\'old\',a.old),(\'new\',Path(state[\'candidate\']))]:\n        command([sys.executable,source/\'record_terrain.py\',\'--checkpoint\',checkpoint,\'--run\',out/label,\'--label\',label],\'record-\'+label+\'.log\')\n    assert (out/\'old/initial-states.json\').read_bytes()==(out/\'new/initial-states.json\').read_bytes()\n    command([\'ffmpeg\',\'-n\',\'-i\',out/\'old/terrain.mp4\',\'-i\',out/\'new/terrain.mp4\',\'-filter_complex\',\'hstack=inputs=2\',\'-c:v\',\'libx264\',\'-pix_fmt\',\'yuv420p\',out/\'comparison.mp4\'],\'compose.log\')\n    import av\n    with av.open(str(out/\'comparison.mp4\')) as video:\n        fps=float(video.streams.video[0].average_rate);frames=sum(1 for _ in video.decode(video=0))\n    assert fps==25 and frames==600,(fps,frames)\n    summary.update(video_fps=fps,video_frames=frames)\n    status(\'exporting_curves\')\n    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator\n    import matplotlib.pyplot as plt\n    rows=[];fig,axes=plt.subplots(2,1,figsize=(10,6))\n    for run in [Path(j[\'run\']) for j in state[\'history\'] if j[\'mode\']==\'train\']:\n        dest=out/\'raw\'/run.name;dest.mkdir(parents=True)\n        shutil.copy2(run/\'run.log\',dest/\'run.log\')\n        for event in run.glob(\'**/events.out.tfevents.*\'):\n            shutil.copy2(event,dest/event.name);acc=EventAccumulator(str(event),size_guidance={\'scalars\':0});acc.Reload()\n            for tag in acc.Tags()[\'scalars\']:\n                values=acc.Scalars(tag);rows.extend(dict(run=run.name,tag=tag,step=v.step,wall_time=v.wall_time,value=v.value) for v in values)\n                if tag in [\'Train/mean_reward\',\'Train/mean_episode_length\']:\n                    axis=axes[0 if tag.endswith(\'mean_reward\') else 1];axis.plot([v.step for v in values],[v.value for v in values],label=run.name);axis.set_ylabel(tag)\n    assert rows,\'No training scalars\'\n    with (out/\'scalars.csv\').open(\'w\') as f:\n        w=csv.DictWriter(f,fieldnames=rows[0]);w.writeheader();w.writerows(rows)\n    for axis in axes:axis.legend(fontsize=7);axis.grid(alpha=.3)\n    fig.tight_layout();fig.savefig(out/\'learning-curves.png\');plt.close(fig);summary[\'scalar_rows\']=len(rows)\n    (out/\'summary.json\').write_text(json.dumps(summary,indent=2))\n    status(\'saving\')\n    hashes={}\n    for path in a.session.rglob(\'*\'):\n        if not path.is_file() or path.name.endswith(\'.partial\') or path.name in [\'status.json\',\'pipeline-status.json\',\'preflight-status.json\']:continue\n        relative=path.relative_to(a.session);target=a.backup/relative;target.parent.mkdir(parents=True,exist_ok=True)\n        shutil.copy2(path,target);digest=hashlib.sha256(path.read_bytes()).hexdigest();assert hashlib.sha256(target.read_bytes()).hexdigest()==digest;hashes[str(relative)]=digest\n    (out/\'file-sha256.json\').write_text(json.dumps(hashes,indent=2));summary[\'verified_files\']=len(hashes)\n    (out/\'summary.json\').write_text(json.dumps(summary,indent=2));status(\'drive_verified_local_snapshot_pending\',**summary)\n    shutil.copytree(out,a.backup/\'final-artifacts\',dirs_exist_ok=True)\n    # Small review bundle excludes model binaries and redundant source revisions.\n    archive=a.session/\'terrain-evidence.zip\'\n    with zipfile.ZipFile(archive,\'x\',zipfile.ZIP_DEFLATED) as z:\n        for path in a.session.rglob(\'*\'):\n            if not path.is_file() or path.suffix in [\'.pt\',\'.zip\',\'.mp4\'] or \'source-before-\' in str(path):continue\n            z.write(path,path.relative_to(a.session))\n    shutil.copy2(archive,a.backup/archive.name);assert hashlib.sha256(archive.read_bytes()).digest()==hashlib.sha256((a.backup/archive.name).read_bytes()).digest()\n    print(json.dumps(summary,indent=2))\nexcept BaseException as e:\n    status(\'failed\',error=f\'{type(e).__name__}: {e}\');raise\n', 'audit_commands.py': '"""Audit saved CSV/video/Drive copies without rerunning learning or physics."""\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport sys\n\n\ndef digest(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef main(session):\n    state = json.loads((session / "status.json").read_text())\n    assert state["phase"] == "complete"\n    destination = Path(state["archive"]).parent\n    result = {"scope": "saved trajectory audit; no physics rerun", "evaluations": []}\n    # Independently spell out the intended schedule; do not reuse score_trial.\n    ends = [3, 7, 11, 15, 19, 23, 27, 30]\n    targets = [(0, 0), (.2, 0), (.7, 0), (0, 0), (0, .5), (0, -.5), (.4, .4), (0, 0)]\n    formal_jobs = [j for j in state["history"] if j["metrics"].get("formal")]\n    assert len(formal_jobs) == 2\n    initial_states = []\n    for job in formal_jobs:\n        run = Path(job["run"])\n        groups = {}\n        with (run / "rollout.csv").open() as stream:\n            for raw in csv.DictReader(stream):\n                row = {k: float(raw[k]) for k in ["time", "vx", "vy", "wz", "cmd_vx", "cmd_yaw"]}\n                row.update(step=int(raw["step"]), done=raw["done"] == "True")\n                assert all(math.isfinite(v) for v in row.values())\n                trial = int(raw["trial"])\n                groups.setdefault(trial, []).append(row)\n        assert len(groups) == 30\n        trials = []\n        for index, saved in enumerate(job["metrics"]["trials"]):\n            rows = groups[index]\n            assert [r["step"] for r in rows] == list(range(len(rows)))\n            segments = []\n            for segment, (vx_target, yaw_target) in enumerate(targets):\n                start = ends[segment-1] if segment else 0\n                lo, hi = round(start * 50), round(ends[segment] * 50)\n                block = rows[lo:hi]\n                yaw_target *= saved["mirror"]\n                for row in block:\n                    assert abs(row["time"] - (row["step"] + 1) / 50) < 1e-8\n                    assert abs(row["cmd_vx"] - vx_target) < 1e-8\n                    assert abs(row["cmd_yaw"] - yaw_target) < 1e-8\n                stable = [r for r in block if r["time"] - start > 1 and not r["done"]]\n                maes = [sum(abs(r[key] - target) for r in stable) / len(stable) if stable else None\n                        for key, target in [("vx", vx_target), ("vy", 0), ("wz", yaw_target)]]\n                streak, settled = 0, None\n                stop = vx_target == yaw_target == 0\n                turn = vx_target == 0 and yaw_target != 0\n                for r in block:\n                    xy = math.hypot(r["vx"], r["vy"])\n                    good = (xy < .05 and abs(r["wz"]) < .1) if stop else (\n                        abs(r["vx"]-vx_target) <= .1 and abs(r["vy"]) <= .1 and abs(r["wz"]-yaw_target) <= .15 and (not turn or xy <= .1))\n                    streak = streak + 1 if good and not r["done"] else 0\n                    if streak >= 25 and settled is None:\n                        settled = r["time"] - start\n                xy_mean = sum(math.hypot(r["vx"], r["vy"]) for r in stable) / len(stable) if stable else None\n                ok = bool(len(block) == hi-lo and stable and not any(r["done"] for r in block)\n                          and all(x <= bound for x, bound in zip(maes, [.1, .1, .15]))\n                          and (not turn or xy_mean <= .1) and settled is not None and settled <= 2)\n                assert ok == saved["segments"][segment]["success"]\n                segments.append({"index": segment, "success": ok, "mae_vx_vy_yaw": maes, "settled_at_s": settled})\n            success = len(rows) == 1500 and all(s["success"] for s in segments) and not any(r["done"] for r in rows)\n            assert success == saved["success"]\n            trials.append({"index": index, "seed": saved["seed"], "yaw_degrees": saved["yaw_degrees"],\n                           "mirror": saved["mirror"], "success": success,\n                           "failed_segments": [s for s in segments if not s["success"]]})\n        initial_states.append(json.loads((run / "initial-states.json").read_text()))\n        result["evaluations"].append({"run": run.name, "successes": sum(t["success"] for t in trials),\n                                      "trials": 30, "failures": [t for t in trials if not t["success"]]})\n    assert initial_states[0] == initial_states[1], "Baseline conditions differ"\n    import av\n    from PIL import Image, ImageDraw\n    recording = Path(state["recording"])\n    video = recording / "commands.mp4"\n    montage = Image.new("RGB", (960, 720))\n    selected = [25, 150, 250, 400, 500, 700]\n    with av.open(str(video)) as container:\n        stream = container.streams.video[0]\n        fps = float(stream.average_rate)\n        count = 0\n        for frame in container.decode(video=0):\n            if count in selected:\n                tile = frame.to_image().resize((480, 240))\n                ImageDraw.Draw(tile).text((5, 220), f"{count/fps:.1f}s", fill="yellow")\n                i = selected.index(count)\n                montage.paste(tile, ((i % 2)*480, (i//2)*240))\n            count += 1\n    assert count == 750 and fps == 25.0, (count, fps)\n    record_metrics = json.loads((recording / "metrics.json").read_text())\n    assert record_metrics["trials"][0]["steps"] == 1500\n    archive = session / Path(state["archive"]).name\n    assert digest(archive) == digest(Path(state["archive"])) == state["archive_sha256"]\n    assert digest(Path(state["checkpoint"])) == digest(Path(state["checkpoint_drive"]))\n    assert digest(video) == digest(destination / "runs" / recording.name / video.name)\n    result.update(video={"frames": count, "fps": fps, "seconds": count/fps, "sha256": digest(video)},\n                  archive_sha256=digest(archive), archive_bytes=archive.stat().st_size,\n                  checkpoint_sha256=digest(Path(state["checkpoint"])),\n                  recording_success=record_metrics["success_rate"], status="passed")\n    audit = session / "final-audit"\n    audit.mkdir(exist_ok=False)\n    (audit / "audit.json").write_text(json.dumps(result, indent=2) + "\\n")\n    shutil.copy2(__file__, audit / "audit_commands.py")\n    montage.save(audit / "preview.jpg")\n    shutil.copytree(audit, destination / "final-audit", dirs_exist_ok=False)\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == "__main__":\n    main(Path(sys.argv[1]))\n', 'export_learning_curves.py': '"""Export existing TensorBoard scalars and plots, without training or simulation."""\nimport csv\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport zipfile\n\n\ndef main(session):\n    os.environ["MPLBACKEND"] = "Agg"\n    import matplotlib.pyplot as plt\n    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator\n    status = json.loads((session / "status.json").read_text())\n    assert status["phase"] == "complete"\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")\n    output = session / (stamp + "-learning-curves")\n    output.mkdir(exist_ok=False)\n    rows, runs, raw_files = [], [], []\n    for job in status["history"]:\n        if job["mode"] != "train":\n            continue\n        run = Path(job["run"])\n        config = json.loads((run / "config.json").read_text())\n        events = sorted(run.glob("**/events.out.tfevents.*"))\n        assert events, f"Missing TensorBoard events: {run}"\n        label = f\'{run.name} ({config["command_stage"]}, yaw={config["cfgs"][2]["reward_scales"]["tracking_ang_vel"]})\'\n        entry = {"run": run.name, "label": label, "events": [], "tags": []}\n        for event in events:\n            accumulator = EventAccumulator(str(event), size_guidance={"scalars": 0})\n            accumulator.Reload()\n            tags = accumulator.Tags()["scalars"]\n            entry["tags"] = sorted(set(entry["tags"]) | set(tags))\n            target = output / "events" / run.name / event.relative_to(run)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(event, target)\n            raw_files.append(target)\n            entry["events"].append(str(target.relative_to(output)))\n            for tag in tags:\n                rows.extend({"run": run.name, "tag": tag, "step": e.step, "wall_time": e.wall_time, "value": e.value}\n                            for e in accumulator.Scalars(tag))\n        runs.append(entry)\n        log = run / "run.log"\n        if log.exists():\n            target = output / "logs" / (run.name + ".log")\n            target.parent.mkdir(exist_ok=True)\n            shutil.copy2(log, target)\n    assert rows\n    with (output / "scalars.csv").open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=["run", "tag", "step", "wall_time", "value"])\n        writer.writeheader()\n        writer.writerows(rows)\n    tags = sorted({r["tag"] for r in rows})\n    # Plot actual recorded tag names; do not infer missing metrics from rewards.\n    wanted = []\n    for pattern in ["mean_reward", "mean_episode_length", "action_std", "value_function", "value_loss"]:\n        found = next((t for t in tags if pattern in t.lower() and t not in wanted), None)\n        if found:\n            wanted.append(found)\n    if not wanted:\n        wanted = tags[:4]\n    fig, axes = plt.subplots(len(wanted), 1, figsize=(12, 3*len(wanted)), squeeze=False)\n    for ax, tag in zip(axes[:, 0], wanted):\n        for run in runs:\n            values = sorted([r for r in rows if r["run"] == run["run"] and r["tag"] == tag], key=lambda r:r["step"])\n            if values:\n                ax.plot([v["step"] for v in values], [v["value"] for v in values], label=run["label"], linewidth=1)\n        ax.set_title(tag)\n        ax.set_xlabel("PPO iteration recorded in each run (optimizer resets keep separate runs)")\n        ax.grid(alpha=.25)\n        ax.legend(fontsize=7)\n    fig.tight_layout()\n    fig.savefig(output / "learning-curves.png", dpi=140)\n    plt.close(fig)\n    manifest = {"created_utc": datetime.now(timezone.utc).isoformat(), "session": str(session),\n                "scope": "existing scalar data; no learning or physics rerun", "scalar_rows": len(rows),\n                "runs": runs, "plotted_tags": wanted,\n                "files": {str(p.relative_to(output)): hashlib.sha256(p.read_bytes()).hexdigest()\n                          for p in output.rglob("*") if p.is_file()}}\n    (output / "manifest.json").write_text(json.dumps(manifest, indent=2))\n    archive = output.with_suffix(".zip")\n    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:\n        for path in output.rglob("*"):\n            if path.is_file():\n                bundle.write(path, path.relative_to(output))\n    destination = Path(status["archive"]).parent\n    shutil.copy2(archive, destination / archive.name)\n    print(json.dumps({"directory": str(output), "archive": str(archive), "drive": str(destination / archive.name),\n                      "sha256": hashlib.sha256(archive.read_bytes()).hexdigest(), "bytes": archive.stat().st_size,\n                      "scalar_rows": len(rows), "training_runs": len(runs), "tags": wanted}, indent=2))\n\n\nif __name__ == "__main__":\n    main(Path(sys.argv[1]))\n', 'probe_push_env.py': '"""Record zero-force trajectories to compare the wrapper against CommandEnv."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,copy,json\nfrom pathlib import Path\nimport torch\nimport genesis as gs\nfrom command_env import CommandEnv\nfrom push_env import PushEnv\nfrom rsl_rl.runners import OnPolicyRunner\np=argparse.ArgumentParser();p.add_argument(\'kind\',choices=[\'command\',\'push\']);p.add_argument(\'--checkpoint\',type=Path,required=True);p.add_argument(\'--output\',type=Path,required=True);a=p.parse_args()\nsaved=json.loads((a.checkpoint.parent/\'config.json\').read_text())\ngs.init(backend=gs.cuda,precision=\'32\',logging_level=\'warning\',seed=81,performance_mode=True)\nenv=CommandEnv(2,copy.deepcopy(saved[\'cfgs\']),evaluation=True) if a.kind==\'command\' else PushEnv(2,copy.deepcopy(saved[\'cfgs\']),evaluation=True)\nrunner=OnPolicyRunner(env,copy.deepcopy(saved[\'train_cfg\']),None,device=gs.device)\nrunner.load(str(a.checkpoint),load_cfg=dict(actor=True,critic=True,optimizer=False,iteration=False),map_location=str(gs.device))\npolicy=runner.get_inference_policy(device=gs.device)\nobs=env.reset_with_yaws([0,90],[91,92]);rows=[]\nwith torch.no_grad():\n    for step in range(100):\n        obs=env.set_commands([[.5,0,0]]*2)\n        obs,_,_,_=env.step(policy(obs))\n        rows.append({k:v.cpu().tolist() for k,v in env.terminal.items()})\nif a.kind==\'push\':\n    assert env.pulses_applied==0\n    env.start_push(torch.tensor([True,False],device=gs.device),.1,1)\n    assert float(env.force[0,1])>0 and float(env.force[1].abs().sum())==0\n    env.reset()\n    assert int(env.remaining.sum())==0 and float(env.force.abs().sum())==0\n    env.reset_with_yaws([0,90],[91,92])\n    env.start_push(torch.ones(2,dtype=torch.bool,device=gs.device),.1,1)\n    assert abs(float(env.force[0,0]))<1e-4 and float(env.force[0,1])>0\n    assert float(env.force[1,0])<0 and abs(float(env.force[1,1]))<1e-4\n    print(\'mass\',env.mass,\'trunk\',env.trunk.name,\'idx\',env.trunk.idx,\'world force\',env.force.cpu().tolist())\na.output.write_text(json.dumps(rows,allow_nan=False))\nprint(\'ZERO FORCE TRAJECTORY SAVED\',a.output)\n', 'terrain_pipeline.py': '"""One bounded adaptation job, dev/flat checks and fixed formal comparison.\n\nRecording and final preservation are a separate required finishing stage.\n"""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,json,subprocess,sys,uuid,time,shutil\nfrom pathlib import Path\nfrom audit_terrain import audit\np=argparse.ArgumentParser();p.add_argument(\'--session\',type=Path,required=True);p.add_argument(\'--old\',type=Path,required=True);p.add_argument(\'--backup\',type=Path,required=True);a=p.parse_args()\nsource=Path(__file__).parent\nassert not (a.session/\'pipeline-status.json\').exists(),\'Inspect existing pipeline before launching\'\nstate=dict(phase=\'waiting_preflight\',history=[],started=time.time(),training_job_cap=1)\ndef save():\n    temp=a.session/\'pipeline-status.json.partial\';temp.write_text(json.dumps(state,indent=2));temp.replace(a.session/\'pipeline-status.json\')\n    a.backup.mkdir(parents=True,exist_ok=True);shutil.copy2(a.session/\'pipeline-status.json\',a.backup/\'pipeline-status.json\')\ndef job(mode,checkpoint,seed,amplitude,extra=()):\n    run=a.session/\'adaptation\'/(f\'{len(state["history"]):02d}-{mode}-\'+uuid.uuid4().hex[:6]);run.parent.mkdir(exist_ok=True)\n    cmd=[sys.executable,str(source/\'terrain_worker.py\'),mode,\'--run\',str(run),\'--checkpoint\',str(checkpoint),\'--map-seed\',str(seed),\'--amplitude\',str(amplitude),\'--calibration\',str(a.session/\'sampler-calibration.json\'),\'--backup\',str(a.backup/\'adaptation\'),*extra]\n    log=run.with_suffix(\'.log\');state.update(current_run=str(run),command=cmd);save()\n    with log.open(\'w\') as f:\n        proc=subprocess.Popen(cmd,stdout=f,stderr=subprocess.STDOUT,cwd=source);state[\'child_pid\']=proc.pid;save()\n        try:code=proc.wait(timeout=1800)\n        except subprocess.TimeoutExpired:\n            proc.terminate();proc.wait(timeout=30);raise\n    if run.exists():shutil.copy2(log,run/\'run.log\');(run/\'exit-code.txt\').write_text(str(code))\n    assert code==0,f\'{mode} exit {code}: {log}\'\n    metrics=json.loads((run/\'metrics.json\').read_text())\n    if mode==\'eval\':audit(run)\n    state[\'history\'].append(dict(run=str(run),mode=mode,seed=seed,amplitude=amplitude,metrics=metrics));save();return run\ntry:\n    save();deadline=time.monotonic()+2400\n    while True:\n        pre=json.loads((a.session/\'preflight-status.json\').read_text())\n        if pre[\'phase\']==\'failed\':raise RuntimeError(pre.get(\'error\'))\n        if pre[\'phase\']==\'preflight_complete_training_pending\':break\n        if time.monotonic()>deadline:raise TimeoutError(\'preflight wait exceeded\')\n        time.sleep(10)\n    for item in pre[\'history\']:\n        if item[\'mode\']==\'eval\':audit(item[\'run\'])\n    baseline=[t for item in pre[\'history\'] if item[\'mode\']==\'eval\' for t in item[\'metrics\'][\'trials\']]\n    state[\'old_dev_successes\']=sum(t[\'success\'] for t in baseline);state[\'dev_count\']=len(baseline)\n    # Cap reduced for the 10:20 UTC outcome-freeze target. No reward search.\n    if state[\'old_dev_successes\']==len(baseline):\n        state.update(phase=\'baseline_ceiling_no_adaptation\',candidate=str(a.old),adaptation_skipped=True);save()\n    else:\n        state[\'phase\']=\'training_weak\';save();trained=job(\'train\',a.old,10,.01,[\'--num-envs\',\'256\',\'--iterations\',\'301\'])\n        state.update(candidate=str(trained/\'checkpoint.pt\'),phase=\'candidate_dev\');save()\n        for amplitude in [.01,.02,.04]:\n            for seed in range(100,103):job(\'eval\',Path(state[\'candidate\']),seed,amplitude,[\'--initial-count\',\'2\'])\n    state[\'phase\']=\'flat_regression\';save()\n    for label,checkpoint in [(\'old\',a.old),(\'new\',Path(state[\'candidate\']))]:\n        if label==\'new\' and state.get(\'adaptation_skipped\'):\n            state[\'new_regression\']=state[\'old_regression\'];state[\'identical_policy_results_reused\']=True;save();continue\n        r=job(\'regression\',checkpoint,0,0)\n        state[label+\'_regression\']=str(r);save()\n    # Candidate is fixed before formal outcomes, even if it underperforms.\n    state[\'phase\']=\'formal\';save();manifest=json.loads((a.session/\'formal-maps/manifest.json\').read_text())\n    for label,checkpoint in [(\'old\',a.old),(\'new\',Path(state[\'candidate\']))]:\n        if label==\'new\' and state.get(\'adaptation_skipped\'):continue\n        state[\'formal_label\']=label;save()\n        for amplitude in [.01,.02,.04]:\n            for seed in range(200,205):\n                run=job(\'eval\',checkpoint,seed,amplitude)\n                cfg=json.loads((run/\'config.json\').read_text())\n                expected=next(m for m in manifest if m[\'seed\']==seed and m[\'amplitude_m\']==amplitude)\n                assert cfg[\'terrain\']==expected,\'formal map changed\'\n                state[\'history\'][-1][\'formal_label\']=label;save()\n    state[\'phase\']=\'terrain_evaluation_complete_flat_record_save_pending\';save()\nexcept BaseException as e:\n    state.update(phase=\'failed\',error=f\'{type(e).__name__}: {e}\');save();raise\n', 'test_commands.py': '"""Meaningful, dependency-free tests for command boundaries and task scoring."""\nimport math\nimport unittest\nfrom commands import CONFIG, cases, command_at, score_trial\n\n\ndef ideal_rows(stage="full", mirror=1):\n    result = []\n    for step in range(1500):\n        segment, c = command_at(step, stage, mirror)\n        result.append({"step": step, "time": (step+1)*0.02, "segment": segment,\n                       "vx": c[0], "vy": 0.0, "wz": c[2], "done": False,\n                       "roll": 0.0, "pitch": 0.0})\n    return result\n\n\nclass CommandsTest(unittest.TestCase):\n    def test_switch_uses_next_observation_boundary(self):\n        self.assertEqual(command_at(149)[1], [0, 0, 0])\n        self.assertEqual(command_at(150)[1], [0.2, 0, 0])\n        self.assertEqual(command_at(750)[1], [0, 0, 0.5])\n        self.assertEqual(command_at(750, mirror=-1)[1], [0, 0, -0.5])\n        with self.assertRaises(ValueError):\n            command_at(1500)\n\n    def test_all_conditions_are_required(self):\n        self.assertTrue(score_trial(ideal_rows())["success"])\n        self.assertFalse(score_trial(ideal_rows()[:-1])["success"])\n        rows = ideal_rows()\n        rows[300]["done"] = True\n        self.assertFalse(score_trial(rows)["success"])\n\n    def test_standing_cannot_pass_motion_or_turns(self):\n        rows = ideal_rows()\n        for r in rows:\n            r.update(vx=0, vy=0, wz=0)\n        metrics = score_trial(rows)\n        self.assertFalse(metrics["success"])\n        self.assertTrue(metrics["segments"][0]["success"])\n        self.assertFalse(metrics["segments"][4]["success"])\n\n    def test_stop_requires_half_second_continuous_dwell(self):\n        rows = ideal_rows()\n        for r in rows:\n            if r["segment"] == 3:\n                r["vx"] = 0 if r["step"] % 24 < 23 else 0.1\n        self.assertIsNone(score_trial(rows)["segments"][3]["settled_at_s"])\n\n    def test_slow_stop_is_failure_even_if_late_mae_is_good(self):\n        rows = ideal_rows()\n        for r in rows:\n            if r["segment"] == 3 and r["time"] < 13.1:\n                r["vx"] = 0.06\n        seg = score_trial(rows)["segments"][3]\n        self.assertLess(seg["vx_mae"], 0.1)\n        self.assertFalse(seg["success"])\n\n    def test_formal_cases_include_both_turns_and_yaws(self):\n        c = cases(formal=True)\n        self.assertEqual(len(c), 30)\n        self.assertEqual({r["yaw_degrees"] for r in c}, {-15, 0, 15})\n        self.assertEqual({r["mirror"] for r in c}, {-1, 1})\n        self.assertTrue(set(CONFIG["formal_seeds"]).isdisjoint({r["seed"] for r in cases()}))\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'push_worker.py': '"""Bounded physical push experiments. Saved command cfg is the only initial cfg."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom command_worker import dump, backup\nfrom pushes import cases, score\n\n\ndef evaluate(env, runner, args):\n    import torch\n    trial_cases=cases(args.formal)\n    obs=env.reset_with_yaws([0]*len(trial_cases),[c[\'seed\'] for c in trial_cases])\n    dump(args.run/\'initial-states.json\',[dict(c,qpos=q) for c,q in zip(trial_cases,env.evaluation_initial_qpos)])\n    policy=runner.get_inference_policy(device=env.device)\n    rows=[[] for _ in trial_cases]\n    active=[True]*len(rows)\n    # Development pushes vary in onset; formal always starts at t=4s.\n    starts=[200 if args.formal else 175+int((c[\'seed\']-90)*18.75) for c in trial_cases]\n    try:\n        with torch.no_grad():\n            for step in range(600):\n                mask=torch.tensor([active[i] and step==s for i,s in enumerate(starts)],device=env.device)\n                env.start_push(mask,[c[\'alpha\'] for c in trial_cases],[c[\'sign\'] for c in trial_cases])\n                obs=env.set_commands([[.5,0,0]]*len(rows))\n                assert torch.allclose(obs[\'policy\'][:,6:9],env.commands*env.commands_scale)\n                action=policy(obs)\n                if not torch.isfinite(action).all(): raise RuntimeError(\'nonfinite action\')\n                obs,_,_,_=env.step(action)\n                state={k:v.detach().cpu().tolist() for k,v in env.terminal.items()}\n                force=env.applied_force.cpu().tolist()\n                actions=action.cpu().tolist()\n                terms={k:v.cpu().tolist() for k,v in env.reward_values.items()}\n                for i,c in enumerate(trial_cases):\n                    if not active[i]:continue\n                    vel=state[\'base_lin_vel\'][i]; ang=state[\'base_ang_vel\'][i]\n                    pos=state[\'base_pos\'][i]; euler=state[\'base_euler\'][i]\n                    done=bool(state[\'reset_buf\'][i])\n                    row=dict(step=step,time=(step+1)*.02,cmd_vx=.5,cmd_vy=0,cmd_wz=0,\n                             vx=vel[0],vy=vel[1],vz=vel[2],wz=ang[2],x=pos[0],y=pos[1],z=pos[2],\n                             roll=euler[0],pitch=euler[1],fx=force[i][0],fy=force[i][1],fz=force[i][2],\n                             reward=state[\'rew_buf\'][i],done=done,\n                             termination=\'solver_error\' if state[\'solver_error\'][i] else (\'roll_pitch_limit\' if done else \'none\'))\n                    row.update({f\'action_{j}\':v for j,v in enumerate(actions[i])})\n                    row.update({f\'reward_{k}\':v[i] for k,v in terms.items()})\n                    row[\'joint_speed_rms\']=math.sqrt(sum(v*v for v in state[\'dof_vel\'][i])/12)\n                    if not all(math.isfinite(v) for v in row.values() if isinstance(v,(float,int))):\n                        raise RuntimeError(\'nonfinite rollout\')\n                    rows[i].append(row)\n                    if done:active[i]=False\n                if not any(active):break\n    finally:\n        nonempty=next((r for r in rows if r),None)\n        if nonempty:\n            with (args.run/\'rollout.csv\').open(\'w\',newline=\'\') as f:\n                w=csv.DictWriter(f,fieldnames=[\'trial\',\'alpha\',\'sign\',\'seed\',*nonempty[0]])\n                w.writeheader()\n                for i,rs in enumerate(rows):\n                    for row in rs:w.writerow(dict(trial=i,**trial_cases[i],**row))\n    trials=[]\n    for c,rs,start in zip(trial_cases,rows,starts):\n        # Shift the dev onset to the common scoring origin (formal is unchanged).\n        shifted=[dict(r,time=r[\'time\']+(200-start)*.02) for r in rs]\n        result=score(shifted)\n        trials.append(dict(c,**result,steps=len(rs),push_start_step=start,\n                           max_roll_pitch=max((max(abs(r[\'roll\']),abs(r[\'pitch\'])) for r in rs),default=None)))\n    return dict(status=\'evaluated\',formal=args.formal,trials=trials,\n                success_rate=sum(t[\'success\'] for t in trials)/len(trials),\n                by_alpha={str(a):sum(t[\'success\'] for t in trials if t[\'alpha\']==a)/sum(t[\'alpha\']==a for t in trials) for a in (.1,.2,.3)})\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'mode\',choices=[\'smoke\',\'train\',\'eval\',\'regression\'])\n    p.add_argument(\'--run\',type=Path,required=True)\n    p.add_argument(\'--checkpoint\',type=Path,required=True)\n    p.add_argument(\'--backup\',type=Path)\n    p.add_argument(\'--resume\',action=\'store_true\')\n    p.add_argument(\'--stage\',choices=[\'weak\',\'mixed\'],default=\'weak\')\n    p.add_argument(\'--formal\',action=\'store_true\')\n    p.add_argument(\'--iterations\',type=int,default=301)\n    p.add_argument(\'--num-envs\',type=int,default=1024)\n    p.add_argument(\'--seed\',type=int,default=81)\n    args=p.parse_args()\n    if not 1<=args.iterations<=301 or not 1<=args.num_envs<=1024:raise ValueError(\'experiment bounds\')\n    args.run.mkdir(parents=True,exist_ok=False)\n    import torch\n    import genesis as gs\n    from push_env import PushEnv\n    from rsl_rl.runners import OnPolicyRunner\n    saved=json.loads((args.checkpoint.parent/\'config.json\').read_text())\n    spec=json.loads(Path(__file__).with_name(\'upstream.lock.json\').read_text())\n    assert saved[\'upstream\']==spec\n    push_config=json.loads(Path(__file__).with_name(\'push-config.json\').read_text())\n    if args.resume and saved.get(\'push_config\')!=push_config:raise ValueError(\'push config changed; cannot resume\')\n    cfgs=copy.deepcopy(saved[\'cfgs\']); train_cfg=copy.deepcopy(saved[\'train_cfg\'])\n    assert cfgs[2][\'reward_scales\'][\'tracking_ang_vel\']==.5,\'must inherit command reward\'\n    assert cfgs[0][\'termination_if_roll_greater_than\']==10\n    assert cfgs[0][\'termination_if_pitch_greater_than\']==10\n    cfgs[0][\'resampling_time_s\']=4.\n    if args.mode in (\'eval\',\'regression\'):cfgs[0][\'episode_length_s\']=35. if args.mode==\'regression\' else 15.\n    if args.resume and (saved.get(\'push_stage\')!=args.stage or saved.get(\'task\')!=\'go2-push-v1\'):\n        raise ValueError(\'resume must preserve task and stage\')\n    train_cfg[\'save_interval\']=25\n    config=dict(task=\'go2-push-v1\',push_config=push_config,push_stage=args.stage,cfgs=cfgs,train_cfg=train_cfg,upstream=spec,\n                initialization_sha256=hashlib.sha256(args.checkpoint.read_bytes()).hexdigest(),\n                args={k:str(v) if isinstance(v,Path) else v for k,v in vars(args).items()})\n    dump(args.run/\'config.json\',config)\n    names=[\'push-config.json\',\'push_worker.py\',\'push_env.py\',\'pushes.py\',\'command_env.py\',\'command_worker.py\',\'commands.py\',\n           \'command-config.json\',\'go2_env.py\',\'go2_record_env.py\',\'go2_train.py\',\'upstream.lock.json\']\n    for name in names:shutil.copy2(Path(__file__).with_name(name),args.run/name)\n    dump(args.run/\'source-sha256.json\',{n:hashlib.sha256((args.run/n).read_bytes()).hexdigest() for n in names})\n    dump(args.run/\'dependencies.json\',{n:importlib.metadata.version(n) for n in [\'genesis-world\',\'torch\',\'rsl-rl-lib\']})\n    (args.run/\'requirements.freeze.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\n    torch.set_num_threads(2)\n    gs.init(backend=gs.cuda,precision=\'32\',logging_level=\'warning\',seed=args.seed,performance_mode=True)\n    evaluation=args.mode in (\'eval\',\'regression\')\n    from commands import cases as command_cases\n    n=(len(command_cases(args.formal,\'full\',False)) if args.mode==\'regression\' else len(cases(args.formal))) if evaluation else (8 if args.mode==\'smoke\' else args.num_envs)\n    env=PushEnv(n,cfgs,training_range=None if evaluation else ((.05,.15) if args.stage==\'weak\' else (.05,.30)),evaluation=evaluation)\n    dump(args.run/\'force-api.json\',dict(mass_kg=env.mass,link_name=env.trunk.name,global_index=env.trunk.idx,ref=\'link_COM\',local=False))\n    class Runner(OnPolicyRunner):\n        def save(self,path,infos=None):\n            super().save(path,infos)\n            backup(args.run,args.backup)\n    runner=Runner(env,train_cfg,str(args.run),device=gs.device)\n    runner.load(str(args.checkpoint),load_cfg=None if args.resume else dict(actor=True,critic=True,optimizer=False,iteration=False),map_location=str(gs.device))\n    if args.resume:runner.current_learning_iteration+=1\n    started=time.monotonic()\n    if args.mode==\'regression\':\n        from command_worker import evaluate as evaluate_commands\n        metrics=evaluate_commands(env,runner,args,\'full\')\n    elif evaluation:metrics=evaluate(env,runner,args)\n    else:\n        # Smoke explicitly exercises a pulse; training uses independent actual-reset clocks.\n        if args.mode==\'smoke\':env.start_push(torch.ones(n,device=gs.device,dtype=torch.bool),.05,1)\n        before=[v.detach().clone() for v in runner.alg.get_policy().parameters()]\n        iterations=2 if args.mode==\'smoke\' else args.iterations\n        runner.learn(num_learning_iterations=iterations,init_at_random_ep_len=True)\n        delta=sum(float((a-b).abs().sum()) for a,b in zip(runner.alg.get_policy().parameters(),before))\n        assert math.isfinite(delta) and delta>0\n        runner.save(str(args.run/\'checkpoint.pt\'))\n        metrics=dict(status=\'ppo_updates_completed\',iterations=iterations,parameter_l1_change=delta,\n                     pulses_applied=env.pulses_applied,optimizer_restored=args.resume)\n    metrics[\'elapsed_s\']=time.monotonic()-started\n    dump(args.run/\'metrics.json\',metrics)\n    backup(args.run,args.backup)\n    print(json.dumps(metrics,indent=2),flush=True)\n\nif __name__==\'__main__\':main()\n', 'summarize_push.py': '"""Descriptive auxiliary metrics from all formal trials; no model selection."""\nimport csv,json,math,statistics,argparse\nfrom pathlib import Path\n\ndef summarize(run):\n    groups={}\n    with (run/\'rollout.csv\').open() as f:\n        for r in csv.DictReader(f):groups.setdefault(int(r[\'trial\']),[]).append(r)\n    results=[]\n    for trial,rows in groups.items():\n        pulse=[r for r in rows if math.hypot(float(r[\'fx\']),float(r[\'fy\']))>0]\n        first=pulse[0]; mag=math.hypot(float(first[\'fx\']),float(first[\'fy\']))\n        axis=[float(first[\'fx\'])/mag,float(first[\'fy\'])/mag]\n        # World displacement along the actual signed push direction, from just before onset.\n        onset=int(first[\'step\']);before=next(r for r in rows if int(r[\'step\'])==onset-1)\n        post=[r for r in rows if 4.2<float(r[\'time\'])<=7.2]\n        lateral=[(float(r[\'x\'])-float(before[\'x\']))*axis[0]+(float(r[\'y\'])-float(before[\'y\']))*axis[1] for r in rows if int(r[\'step\'])>=onset]\n        changes=[sum((float(b[f\'action_{j}\'])-float(a[f\'action_{j}\']))**2 for j in range(12))/12 for a,b in zip(rows,rows[1:])]\n        results.append(dict(trial=trial,alpha=float(first[\'alpha\']),seed=int(first[\'seed\']),sign=int(first[\'sign\']),\n            velocity_error_integral_3s_m=sum(math.hypot(float(r[\'vx\'])-.5,float(r[\'vy\']))*.02 for r in post),\n            post_window_steps=len(post),max_abs_roll_pitch_deg=max(max(abs(float(r[\'roll\'])),abs(float(r[\'pitch\']))) for r in rows),\n            max_abs_push_axis_displacement_m=max(map(abs,lateral)),final_push_axis_displacement_m=lateral[-1],\n            joint_velocity_rms_rad_s=math.sqrt(statistics.mean(float(r[\'joint_speed_rms\'])**2 for r in rows)),\n            action_step_delta_rms=math.sqrt(statistics.mean(changes)),terminated=any(r[\'done\']==\'True\' for r in rows)))\n    assert len(results)==60 and all(r[\'post_window_steps\']==150 for r in results)\n    fields=[\'velocity_error_integral_3s_m\',\'max_abs_roll_pitch_deg\',\'max_abs_push_axis_displacement_m\',\'final_push_axis_displacement_m\',\'joint_velocity_rms_rad_s\',\'action_step_delta_rms\']\n    return dict(trials=results,by_alpha={str(a):{k:dict(median=statistics.median(r[k] for r in results if r[\'alpha\']==a),max=max(r[k] for r in results if r[\'alpha\']==a)) for k in fields} for a in (.1,.2,.3)},scope=\'All 60 fixed formal trials; descriptive only; lateral displacement uses signed applied-force axis, not path restoration.\')\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--session\',type=Path,required=True);a=p.parse_args()\n    state=json.loads((a.session/\'status.json\').read_text())\n    result={n:summarize(Path(state[n+\'_formal\'])) for n in (\'old\',\'new\')}\n    target=a.session/\'final-artifacts/auxiliary-metrics.json\';target.write_text(json.dumps(result,indent=2));print(target)\n', 'record_push.py': '"""Two prespecified 12s demonstrations (left/right), with measured telemetry overlay."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,copy,csv,json,math\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport genesis as gs\nimport av\nfrom PIL import Image,ImageDraw\nfrom push_env import PushEnv\nfrom rsl_rl.runners import OnPolicyRunner\np=argparse.ArgumentParser();p.add_argument(\'--checkpoint\',type=Path,required=True);p.add_argument(\'--run\',type=Path,required=True);p.add_argument(\'--label\',required=True);a=p.parse_args()\na.run.mkdir(parents=True,exist_ok=False)\nsaved=json.loads((a.checkpoint.parent/\'config.json\').read_text());cfgs=copy.deepcopy(saved[\'cfgs\']);cfgs[0][\'episode_length_s\']=15\n(torch.set_num_threads(2))\ngs.init(backend=gs.cuda,precision=\'32\',logging_level=\'warning\',seed=81,performance_mode=True)\nenv=PushEnv(1,cfgs,evaluation=True,record=True)\nrunner=OnPolicyRunner(env,copy.deepcopy(saved[\'train_cfg\']),None,device=gs.device)\nrunner.load(str(a.checkpoint),load_cfg=dict(actor=True,critic=True,optimizer=False,iteration=False),map_location=str(gs.device))\npolicy=runner.get_inference_policy(device=gs.device)\nvideo=av.open(str(a.run/\'push.mp4\'),\'w\');stream=video.add_stream(\'libx264\',rate=25)\nstream.width,stream.height,stream.pix_fmt=640,480,\'yuv420p\'\nrows=[];initial=[]\ntry:\n    for sign in (-1,1):\n        env.reset_with_yaws([0],[2000]);initial.append(env.evaluation_initial_qpos)\n        terminated=False; streak=0; recovery=None\n        with torch.no_grad():\n            for step in range(600):\n                if step==200 and not terminated:env.start_push(torch.tensor([True],device=gs.device),.3,sign)\n                obs=env.set_commands([[.5,0,0]])\n                env.step(policy(obs))\n                t=env.terminal\n                vel=t[\'base_lin_vel\'][0].cpu().tolist();wz=float(t[\'base_ang_vel\'][0,2]);pos=t[\'base_pos\'][0].cpu().numpy()\n                done=bool(t[\'reset_buf\'][0]);terminated=terminated or done\n                force=env.applied_force[0].cpu().tolist()\n                good=not terminated and abs(vel[0]-.5)<=.1 and abs(vel[1])<=.1 and abs(wz)<=.15\n                if step>=210:\n                    streak=streak+1 if good else 0\n                    if streak>=25 and recovery is None:recovery=(step+1-210)*.02\n                row=dict(sign=sign,step=step,time=(step+1)*.02,vx=vel[0],vy=vel[1],wz=wz,fx=force[0],fy=force[1],done=done,terminated=terminated)\n                rows.append(row)\n                if step%2==0:\n                    env.camera.set_pose(pos=pos+np.array([1.8,-2.,1.3]),lookat=pos)\n                    picture=Image.fromarray(env.camera.render()[0]);draw=ImageDraw.Draw(picture)\n                    draw.rectangle((0,0,640,83),fill=\'black\')\n                    draw.text((8,4),f\'{a.label} | {"LEFT" if sign==1 else "RIGHT"} | t={(step+1)*.02:.2f}s | playback 1x\',fill=\'white\')\n                    draw.text((8,22),f\'command vx=.50 vy=0 yaw=0 | actual {vel[0]:.2f} {vel[1]:.2f} {wz:.2f}\',fill=\'white\')\n                    magnitude=math.hypot(force[0],force[1])\n                    draw.text((8,40),f\'push alpha=.30 | applied {magnitude:.1f}N | \'+(\'PUSHING\' if magnitude>0 else \'force off\'),fill=\'yellow\' if magnitude else \'white\')\n                    draw.text((8,58),f\'recovery (25 good steps): {recovery if recovery is not None else "pending"} s | terminated={terminated}\',fill=\'white\')\n                    if magnitude:\n                        start=(320,180);end=(320+sign*90,180)\n                        draw.line((start,end),fill=\'red\',width=7)\n                        draw.polygon((end,(end[0]-sign*18,end[1]-10),(end[0]-sign*18,end[1]+10)),fill=\'red\')\n                    for packet in stream.encode(av.VideoFrame.from_ndarray(np.asarray(picture),format=\'rgb24\')):video.mux(packet)\nfinally:\n    for packet in stream.encode():video.mux(packet)\n    video.close()\n    if rows:\n        with (a.run/\'record.csv\').open(\'w\') as f:\n            w=csv.DictWriter(f,fieldnames=rows[0]);w.writeheader();w.writerows(rows)\n    (a.run/\'initial-states.json\').write_text(json.dumps(initial))\nprint(\'Recorded\',len(rows),\'steps\',a.run)\n', 'calibrate_terrain.py': '"""Contact-height and two-environment calibration on a known planar height field."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport genesis as gs\np=argparse.ArgumentParser();p.add_argument(\'--run\',type=Path,required=True);a=p.parse_args()\na.run.mkdir(parents=True,exist_ok=False)\ngs.init(backend=gs.cuda,precision=\'32\',logging_level=\'warning\',seed=91)\nscene=gs.Scene(sim_options=gs.options.SimOptions(dt=.02,substeps=2),show_viewer=False)\naxis=np.arange(41)*.1\n# Surface world z=.15+.05*x+.025*y, morph origin (-2,-2,0).\nx,y=np.meshgrid(axis-2,axis-2,indexing=\'ij\');field=(.15+.05*x+.025*y)/.001\nterrain=scene.add_entity(gs.morphs.Terrain(height_field=field,horizontal_scale=.1,vertical_scale=.001,pos=(-2,-2,0)))\nball=scene.add_entity(gs.morphs.Sphere(radius=.05,pos=(0,0,.6)),material=gs.materials.Rigid(rho=1000))\nscene.build(n_envs=2)\npositions=torch.tensor([[0.,0.,.6],[.6,-.4,.6]],device=gs.device)\nball.set_pos(positions,zero_velocity=True)\nfor _ in range(150):scene.step()\nobserved=ball.get_pos().cpu().numpy()\n# Sphere on a plane rests along the normal: z-center at fixed x,y equals plane height + r*sqrt(1+a²+b²).\nexpected=.15+.05*observed[:,0]+.025*observed[:,1]+.05*np.sqrt(1+.05**2+.025**2)\nerror=np.abs(observed[:,2]-expected)\nresult=dict(observed=observed.tolist(),expected_z=expected.tolist(),error_m=error.tolist(),\n            finite=bool(np.isfinite(observed).all()),height_field_shape=list(field.shape))\n(a.run/\'calibration.json\').write_text(json.dumps(result,indent=2))\nassert result[\'finite\'] and float(error.max())<.005,result\nprint(json.dumps(dict(status=\'passed\',**result),indent=2))\n', 'prepare_terrain_retry.py': '"""Build an isolated retry revision; never mutate prior sources/results."""\nimport argparse,shutil\nfrom pathlib import Path\np=argparse.ArgumentParser();p.add_argument(\'--source\',type=Path,required=True);p.add_argument(\'--output\',type=Path,required=True);a=p.parse_args()\nshutil.copytree(a.source,a.output)\nw=(a.source/\'terrain_worker.py\').read_text()\nanchor="p.add_argument(\'--backup\',type=Path)"\nassert w.count(anchor)==1\nw=w.replace(anchor,anchor+"\\np.add_argument(\'--learning-rate\',type=float)\\np.add_argument(\'--posture-scale\',type=float)\\np.add_argument(\'--initial-seed\',type=int,default=3000)")\nanchor="if a.mode==\'regression\':cfgs[0][\'episode_length_s\']=35."\nassert w.count(anchor)==1\nw=w.replace(anchor,anchor+"\\nif a.learning_rate is not None:\\n    assert 0<a.learning_rate<=.001\\n    tcfg[\'algorithm\'][\'learning_rate\']=a.learning_rate\\n    tcfg[\'algorithm\'][\'schedule\']=\'fixed\'\\nif a.posture_scale is not None:\\n    assert -2<=a.posture_scale<=0\\n    if a.posture_scale:cfgs[2][\'reward_scales\'][\'posture\']=a.posture_scale\\n    else:cfgs[2][\'reward_scales\'].pop(\'posture\',None)")\nw=w.replace(\'list(range(3000,3000+n))\',\'list(range(a.initial_seed,a.initial_seed+n))\')\n(a.output/\'terrain_worker.py\').write_text(w)\ne=(a.source/\'terrain_env.py\').read_text()\ne+=\'\\n    def _reward_posture(self):\\n        # Euler buffer is degrees; penalize squared roll/pitch in radians.\\n        return torch.square(torch.deg2rad(self.base_euler[:,:2])).sum(dim=1)\\n\'\n(a.output/\'terrain_env.py\').write_text(e)\nr=(a.source/\'record_terrain.py\').read_text()\nr=r.replace(\'generate(200,.04)\',\'generate(400,.04)\').replace(\'[3000,3001]\',\'[5000,5001]\').replace(\'seed200\',\'seed400\')\n(a.output/\'record_terrain.py\').write_text(r)\n# The flat CommandEnv has no terrain-only posture reward; inference does not use rewards.\nw=(a.output/\'terrain_worker.py\').read_text()\nanchor="if a.mode==\'regression\':cfgs[0][\'episode_length_s\']=35."\nw=w.replace(anchor,anchor+"\\nif a.mode==\'regression\':cfgs[2][\'reward_scales\'].pop(\'posture\',None)")\n(a.output/\'terrain_worker.py\').write_text(w)\n', 'build_terrain_env.py': '"""Create isolated patched copies, preserving original locomotion/command sources."""\nimport argparse,hashlib,json,difflib\nfrom pathlib import Path\np=argparse.ArgumentParser();p.add_argument(\'--source\',type=Path,required=True);p.add_argument(\'--output\',type=Path,required=True);a=p.parse_args();a.output.mkdir(exist_ok=True,parents=True)\nsrc=(a.source/\'go2_record_env.py\').read_text()\nstart=src.index(\'        # add plain\');end=src.index(\'        self.robot =\',start)\nreplacement=\'\'\'        # Terrain-only collision ground; no hidden plane beneath valleys.\n        import numpy as np\n        terrain_data = np.load(env_cfg[\'_terrain_file\'])\n        self.terrain = self.scene.add_entity(gs.morphs.Terrain(\n            height_field=terrain_data[\'height\'], horizontal_scale=0.1,\n            vertical_scale=0.001, pos=(-12., -12., 0.)))\n\n\'\'\'\nnew=src[:start]+replacement+src[end:]\nanchor=\'        # Compute timeout\'\nassert new.count(anchor)==1\nnew=new.replace(anchor,"        self.reset_buf |= (torch.abs(self.base_pos[:, :2]) > 11.5).any(dim=1)\\n\\n"+anchor)\n\n(a.output/\'go2_terrain_base.py\').write_text(new)\ncommand=(a.source/\'command_env.py\').read_text();assert command.count(\'from go2_record_env import Go2Env\')==1\n(a.output/\'terrain_command_base.py\').write_text(command.replace(\'from go2_record_env import Go2Env\',\'from go2_terrain_base import Go2Env\'))\n(a.output/\'terrain-base.patch\').write_text(\'\'.join(difflib.unified_diff(src.splitlines(True),new.splitlines(True),fromfile=\'go2_record_env.py\',tofile=\'go2_terrain_base.py\')))\n(a.output/\'terrain-build-sha256.json\').write_text(json.dumps({n:hashlib.sha256((a.output/n).read_bytes()).hexdigest() for n in [\'go2_terrain_base.py\',\'terrain_command_base.py\']},indent=2))\n', 'test_pushes.py': "import math\nimport unittest\nfrom pushes import lateral_force, pulse_active, cases, score\n\n\nclass PushTests(unittest.TestCase):\n    def test_pulse(self):\n        self.assertEqual(sum(pulse_active(i) for i in range(600)), 10)\n        self.assertFalse(pulse_active(210))\n    def test_frame(self):\n        f=lateral_force(math.pi/2, 1, .2, 10)\n        self.assertAlmostEqual(f[0], -19.62)\n        self.assertAlmostEqual(f[1], 0)\n        self.assertEqual(f[2], 0)\n    def test_cases(self):\n        self.assertEqual(len(cases(True)),60)\n        self.assertFalse({c['seed'] for c in cases(True)} & {c['seed'] for c in cases()})\n    def test_recovery(self):\n        rows=[dict(time=(i+1)*.02,vx=.5,vy=0,wz=0,done=False) for i in range(600)]\n        self.assertTrue(score(rows)['success'])\n        rows[599]['done']=True\n        self.assertFalse(score(rows)['success'])\n        self.assertFalse(score(rows[:400])['success'])\n    def test_dwell(self):\n        rows=[dict(time=(i+1)*.02,vx=.5,vy=.2 if i%20==0 else 0,wz=0,done=False) for i in range(600)]\n        self.assertIsNone(score(rows)['recovery_s'])\n\n\nif __name__ == '__main__': unittest.main()\n", 'terrain_worker.py': '"""Isolated bounded terrain training/evaluation; calibration gate required."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,copy,csv,hashlib,json,shutil,time,importlib.metadata,subprocess,sys\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport genesis as gs\nfrom terrain_maps import generate\nfrom terrain_env import TerrainEnv\nfrom rsl_rl.runners import OnPolicyRunner\np=argparse.ArgumentParser();p.add_argument(\'mode\',choices=[\'smoke\',\'train\',\'eval\',\'regression\']);p.add_argument(\'--run\',type=Path,required=True);p.add_argument(\'--checkpoint\',type=Path,required=True);p.add_argument(\'--map-seed\',type=int,required=True);p.add_argument(\'--amplitude\',type=float,required=True);p.add_argument(\'--num-envs\',type=int,default=256);p.add_argument(\'--iterations\',type=int,default=301);p.add_argument(\'--resume\',action=\'store_true\');p.add_argument(\'--calibration\',type=Path,required=True)\np.add_argument(\'--backup\',type=Path)\np.add_argument(\'--initial-count\',type=int,choices=[2,4],default=4)\n# Parse again after adding the optional preservation destination.\na=p.parse_args()\nassert json.loads(a.calibration.read_text())[\'status\']==\'passed\',\'calibration gate\'\nassert 1<=a.num_envs<=1024 and 1<=a.iterations<=301 and a.amplitude in [0,.01,.02,.04]\na.run.mkdir(parents=True,exist_ok=False)\nsaved=json.loads((a.checkpoint.parent/\'config.json\').read_text());cfgs=copy.deepcopy(saved[\'cfgs\']);tcfg=copy.deepcopy(saved[\'train_cfg\'])\nassert cfgs[2][\'reward_scales\'][\'tracking_ang_vel\']==.5\nassert cfgs[0][\'termination_if_roll_greater_than\']==cfgs[0][\'termination_if_pitch_greater_than\']==10\ncfgs[0][\'episode_length_s\']=15.;tcfg[\'save_interval\']=25\nif a.mode==\'regression\':cfgs[0][\'episode_length_s\']=35.\nheight,meta=generate(a.map_seed,a.amplitude);np.savez(a.run/\'terrain.npz\',height=height)\nif a.resume:assert saved[\'terrain\']==meta,\'resume terrain mismatch\'\nconfig=dict(task=\'go2-terrain-v1\',cfgs=cfgs,train_cfg=tcfg,terrain=meta,upstream=saved[\'upstream\'],initialization_sha256=hashlib.sha256(a.checkpoint.read_bytes()).hexdigest(),args={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()})\n(a.run/\'config.json\').write_text(json.dumps(config,indent=2))\nfor f in Path(__file__).parent.glob(\'*.py\'):shutil.copy2(f,a.run/f.name)\nshutil.copy2(a.calibration,a.run/\'calibration.json\')\n(a.run/\'dependencies.json\').write_text(json.dumps({k:importlib.metadata.version(k) for k in [\'genesis-world\',\'rsl-rl-lib\',\'torch\',\'quadrants\']},indent=2))\n(a.run/\'requirements.freeze.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\ndef backup():\n    if a.backup:\n        destination=a.backup/a.run.name;destination.mkdir(parents=True,exist_ok=True)\n        for f in a.run.rglob(\'*\'):\n            if not f.is_file() or f.name.endswith(\'.partial\'):continue\n            target=destination/f.relative_to(a.run);target.parent.mkdir(parents=True,exist_ok=True)\n            temp=target.with_name(target.name+\'.partial\');shutil.copy2(f,temp);temp.replace(target)\nclass SavedRunner(OnPolicyRunner):\n    def save(self,path,infos=None):\n        super().save(path,infos);backup()\ntorch.set_num_threads(2);gs.init(backend=gs.cuda,precision=\'32\',logging_level=\'warning\',seed=81,performance_mode=True)\nn=a.initial_count if a.mode==\'eval\' else (8 if a.mode==\'smoke\' else a.num_envs)\nif a.mode==\'regression\':\n    from command_env import CommandEnv\n    from commands import cases\n    a.formal=True;n=len(cases(True,\'full\',False))\n    env=CommandEnv(n,cfgs,stage=\'full\',evaluation=True)\nelse:env=TerrainEnv(n,cfgs,a.run/\'terrain.npz\',evaluation=a.mode==\'eval\')\nrunner=SavedRunner(env,tcfg,str(a.run),device=gs.device)\nrunner.load(str(a.checkpoint),load_cfg=None if a.resume else dict(actor=True,critic=True,optimizer=False,iteration=False),map_location=str(gs.device))\nstarted=time.monotonic()\nif a.mode==\'regression\':\n    from command_worker import evaluate\n    metrics=evaluate(env,runner,a,\'full\')\nelif a.mode!=\'eval\':\n    if a.resume:runner.current_learning_iteration+=1\n    before=[x.detach().clone() for x in runner.alg.get_policy().parameters()]\n    count=2 if a.mode==\'smoke\' else a.iterations\n    runner.learn(num_learning_iterations=count,init_at_random_ep_len=True)\n    delta=sum(float((x-y).abs().sum()) for x,y in zip(runner.alg.get_policy().parameters(),before));assert np.isfinite(delta) and delta>0\n    runner.save(str(a.run/\'checkpoint.pt\'));metrics=dict(status=\'ppo_updates_completed\',iterations=count,parameter_l1_change=delta,transitions=n*tcfg[\'num_steps_per_env\']*count)\nelse:\n    env.reset_with_yaws([0]*n,list(range(3000,3000+n)));initial=env.evaluation_initial_qpos\n    (a.run/\'initial-states.json\').write_text(json.dumps(initial))\n    policy=runner.get_inference_policy(device=gs.device);active=[True]*n;rows=[[] for _ in range(n)]\n    with torch.no_grad():\n        for step in range(600):\n            command=[0,0,0] if step<50 else [.5,0,0]\n            obs=env.set_commands([command]*n);actions=policy(obs);env.step(actions);t=env.terminal\n            for i in range(n):\n                if not active[i]:continue\n                pos=t[\'base_pos\'][i].cpu().tolist();vel=t[\'base_lin_vel\'][i].cpu().tolist();angles=t[\'base_euler\'][i].cpu().tolist()\n                row=dict(trial=i,step=step,time=(step+1)*.02,cmd_vx=command[0],x=pos[0],y=pos[1],z=pos[2],vx=vel[0],vy=vel[1],wz=float(t[\'base_ang_vel\'][i,2]),roll=angles[0],pitch=angles[1],done=bool(t[\'reset_buf\'][i]))\n                row[\'ground_height\']=float(env.ground_height(t[\'base_pos\'][:,:2])[i])\n                row[\'reward\']=float(t[\'rew_buf\'][i]);row[\'solver_error\']=bool(t[\'solver_error\'][i])\n                row[\'outside_map\']=max(abs(pos[0]),abs(pos[1]))>11.5\n                row.update({f\'action_{j}\':float(actions[i,j]) for j in range(12)})\n                row.update({f\'joint_velocity_{j}\':float(t[\'dof_vel\'][i,j]) for j in range(12)})\n                assert all(np.isfinite(v) for v in row.values());rows[i].append(row)\n                if row[\'done\']:active[i]=False\n    with (a.run/\'rollout.csv\').open(\'w\') as f:\n        w=csv.DictWriter(f,fieldnames=rows[0][0]);w.writeheader();w.writerows(r for rs in rows for r in rs)\n    trials=[]\n    for i,rs in enumerate(rows):\n        window=[r for r in rs if r[\'time\']>2];mae={k:sum(abs(r[k]-(.5 if k==\'vx\' else 0)) for r in window)/len(window) if window else None for k in [\'vx\',\'vy\',\'wz\']}\n        completed=len(rs)==600 and not any(r[\'done\'] or r[\'solver_error\'] or r[\'outside_map\'] for r in rs)\n        distance=rs[-1][\'x\']-initial[i][0];lateral=max(abs(r[\'y\']-initial[i][1]) for r in rs)\n        success=completed and distance>4 and lateral<.75 and mae[\'vx\']<=.15 and mae[\'vy\']<=.15 and mae[\'wz\']<=.2\n        trials.append(dict(trial=i,success=success,completed=completed,distance=distance,lateral=lateral,mae=mae))\n    metrics=dict(status=\'evaluated\',trials=trials,success_rate=sum(t[\'success\'] for t in trials)/n)\nmetrics[\'elapsed_s\']=time.monotonic()-started\n(a.run/\'metrics.json\').write_text(json.dumps(metrics,indent=2));print(json.dumps(metrics,indent=2))\nbackup()\n', 'terrain_retry_pipeline.py': '"""Two prespecified candidates; fresh holdout, same success criteria."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,json,subprocess,sys,time,uuid,shutil\nfrom pathlib import Path\nimport numpy as np\nfrom terrain_maps import generate\nfrom audit_terrain import audit\np=argparse.ArgumentParser();p.add_argument(\'--session\',type=Path,required=True);p.add_argument(\'--backup\',type=Path,required=True);p.add_argument(\'--old\',type=Path,required=True);p.add_argument(\'--calibration\',type=Path,required=True);a=p.parse_args()\nsrc=Path(__file__).parent;spec=json.loads((src/\'terrain_retry_spec.json\').read_text());a.session.mkdir(parents=True,exist_ok=False)\nstate=dict(phase=\'freezing\',history=[],started=time.time(),training_job_cap=6,spec=spec)\ndef save():\n t=a.session/\'pipeline-status.json.partial\';t.write_text(json.dumps(state,indent=2));t.replace(a.session/\'pipeline-status.json\');a.backup.mkdir(parents=True,exist_ok=True);shutil.copy2(a.session/\'pipeline-status.json\',a.backup/\'pipeline-status.json\')\ndef job(mode,cp,seed,amp,extra=()):\n assert time.time()-state[\'started\']<spec[\'time_cap_seconds\'],\'time cap\'\n run=a.session/\'adaptation\'/(f\'{len(state["history"]):02d}-{mode}-\'+uuid.uuid4().hex[:6]);run.parent.mkdir(exist_ok=True)\n cmd=[sys.executable,str(src/\'terrain_worker.py\'),mode,\'--run\',str(run),\'--checkpoint\',str(cp),\'--map-seed\',str(seed),\'--amplitude\',str(amp),\'--calibration\',str(a.calibration),\'--backup\',str(a.backup/\'adaptation\'),*map(str,extra)]\n state.update(current_run=str(run),command=cmd);save()\n with run.with_suffix(\'.log\').open(\'w\') as f:\n  proc=subprocess.Popen(cmd,stdout=f,stderr=subprocess.STDOUT,cwd=src);state[\'child_pid\']=proc.pid;save()\n  try:code=proc.wait(timeout=1200)\n  except subprocess.TimeoutExpired:proc.terminate();proc.wait(timeout=30);raise\n if run.exists():shutil.copy2(run.with_suffix(\'.log\'),run/\'run.log\');(run/\'exit-code.txt\').write_text(str(code))\n assert code==0,(code,run)\n m=json.loads((run/\'metrics.json\').read_text())\n if mode==\'eval\':audit(run)\n state[\'history\'].append(dict(run=str(run),mode=mode,seed=seed,amplitude=amp,metrics=m));save();return run\ntry:\n formal=a.session/\'formal-maps\';formal.mkdir();manifest=[]\n for amp in spec[\'amplitudes\']:\n  for seed in spec[\'formal_seeds\']:\n   h,meta=generate(seed,amp);np.savez(formal/f\'{seed}-{amp}.npz\',height=h);manifest.append(meta)\n (formal/\'manifest.json\').write_text(json.dumps(manifest,indent=2));shutil.copytree(formal,a.backup/\'formal-maps\')\n scores=[]\n # Smoke uses the new reward and fixed learning rate, isolated from real candidates.\n state[\'phase\']=\'smoke\';save();job(\'smoke\',a.old,11,.02,[\'--learning-rate\',.0001,\'--posture-scale\',-.5])\n for c in spec[\'candidates\']:\n  state.update(phase=\'training\',candidate_name=c[\'name\']);save();cp=a.old\n  for seed,amp in spec[\'training_maps\']:\n   r=job(\'train\',cp,seed,amp,[\'--learning-rate\',c[\'learning_rate\'],\'--posture-scale\',c[\'posture_scale\'],\'--num-envs\',256,\'--iterations\',301]);cp=r/\'checkpoint.pt\'\n  state[\'phase\']=\'candidate_dev\';save();trials=[]\n  for amp in spec[\'amplitudes\']:\n   for seed in spec[\'dev_seeds\']:\n    r=job(\'eval\',cp,seed,amp,[\'--initial-count\',2]);trials.extend((amp,t) for t in json.loads((r/\'metrics.json\').read_text())[\'trials\'])\n  flat=job(\'regression\',cp,0,0);fm=json.loads((flat/\'metrics.json\').read_text());fs=sum(t[\'success\'] for t in fm[\'trials\'])\n  scores.append(dict(name=c[\'name\'],checkpoint=str(cp),dev=sum(t[\'success\'] for _,t in trials),strong=sum(t[\'success\'] for amp,t in trials if amp==.04),flat=fs,regression=str(flat)))\n  state[\'candidates\']=scores;save()\n eligible=[c for c in scores if c[\'flat\']>=27 and c[\'dev\']>=17]\n chosen=max(eligible,key=lambda c:(c[\'dev\'],c[\'strong\'])) if eligible else None\n state.update(candidate=chosen[\'checkpoint\'] if chosen else str(a.old),selection=chosen,phase=\'flat_regression\');save()\n oldflat=job(\'regression\',a.old,0,0);state[\'old_regression\']=str(oldflat)\n state[\'new_regression\']=chosen[\'regression\'] if chosen else str(oldflat);save()\n state[\'phase\']=\'formal\';save()\n for label,cp in [(\'old\',a.old),(\'new\',Path(state[\'candidate\']))]:\n  for amp in spec[\'amplitudes\']:\n   for seed in spec[\'formal_seeds\']:\n    r=job(\'eval\',cp,seed,amp,[\'--initial-seed\',5000])\n    cfg=json.loads((r/\'config.json\').read_text());assert cfg[\'terrain\']==next(m for m in manifest if m[\'seed\']==seed and m[\'amplitude_m\']==amp)\n    state[\'history\'][-1][\'formal_label\']=label;save()\n state[\'phase\']=\'terrain_evaluation_complete_flat_record_save_pending\';save()\nexcept BaseException as e:\n state.update(phase=\'failed\',error=f\'{type(e).__name__}: {e}\');save();raise\n', 'command_env.py': '"""Extend the SHA-verified official environment without editing its source."""\nimport copy\nimport random\nimport torch\nimport genesis as gs\nfrom genesis.utils.geom import inv_quat, transform_by_quat, quat_to_xyz\nfrom go2_record_env import Go2Env\nfrom commands import CONFIG\n\n\nclass CommandEnv(Go2Env):\n    def __init__(self, num_envs, cfgs, stage="full", evaluation=False, record=False):\n        self.stage = copy.deepcopy(CONFIG["stages"][stage])\n        self.evaluation = evaluation\n        self._inside_step = False\n        self.terminal = None\n        super().__init__(num_envs, *copy.deepcopy(cfgs), record=record)\n        self.reward_values = {}\n        if evaluation:\n            # Capture the actual old-command rewards, before upstream resampling.\n            for name, function in list(self.reward_functions.items()):\n                def wrapped(fn=function, key=name):\n                    value = fn()\n                    self.reward_values[key] = (value * self.reward_scales[key]).detach().clone()\n                    return value\n                self.reward_functions[name] = wrapped\n\n    def _resample_commands(self, envs_idx):\n        if self.evaluation:\n            # Evaluation commands are injected explicitly, before observation/policy.\n            return\n        n = self.num_envs\n        modes = torch.multinomial(torch.tensor(self.stage["weights"], device=self.device), n, replacement=True)\n        vx = 0.1 + torch.rand(n, device=self.device) * (self.stage["vx_max"] - 0.1)\n        arc_vx = 0.1 + torch.rand(n, device=self.device) * (self.stage["arc_vx_max"] - 0.1)\n        yaw = (0.2 + torch.rand(n, device=self.device) * max(0.0, self.stage["yaw_max"] - 0.2))\n        yaw *= torch.where(torch.rand(n, device=self.device) < 0.5, -1.0, 1.0)\n        values = torch.zeros_like(self.commands)\n        values[:, 0] = torch.where(modes == 1, vx, torch.where(modes == 3, arc_vx, 0.0))\n        values[:, 2] = torch.where((modes == 2) | (modes == 3), yaw, 0.0)\n        if envs_idx is None:\n            self.commands.copy_(values)\n        else:\n            torch.where(envs_idx[:, None], values, self.commands, out=self.commands)\n\n    def set_commands(self, values):\n        values = torch.as_tensor(values, device=self.device, dtype=gs.tc_float)\n        if values.shape != self.commands.shape or not torch.isfinite(values).all():\n            raise ValueError("Invalid commands")\n        self.commands.copy_(values)\n        self._update_observation()\n        return self.get_observations()\n\n    def _reset_idx(self, envs_idx=None):\n        if self.evaluation and self._inside_step:\n            # Upstream updates these buffers, then calls this hook before resetting.\n            self.terminal = {k: getattr(self, k).detach().clone() for k in\n                             ["base_pos", "base_quat", "base_euler", "base_lin_vel", "base_ang_vel", "dof_vel", "reset_buf", "rew_buf"]}\n            self.terminal["solver_error"] = self.scene.rigid_solver.get_error_envs_mask().clone()\n        super()._reset_idx(envs_idx)\n        if self.evaluation:\n            if envs_idx is None:\n                self.commands.zero_()\n            else:\n                self.commands.masked_fill_(envs_idx[:, None], 0.0)\n\n    def step(self, actions):\n        self._inside_step = True\n        try:\n            return super().step(actions)\n        finally:\n            self._inside_step = False\n\n    def reset_with_yaws(self, yaw_degrees, seeds):\n        self.reset()\n        angles = torch.deg2rad(torch.tensor(yaw_degrees, device=self.device, dtype=gs.tc_float))\n        qpos = self.init_qpos.repeat(self.num_envs, 1)\n        qpos[:, 3] = torch.cos(angles / 2)\n        qpos[:, 4:6] = 0\n        qpos[:, 6] = torch.sin(angles / 2)\n        # Deterministic policy evaluation would otherwise repeat identical rollouts\n        # for every seed. Each seed defines a small, recorded initial joint offset.\n        offsets = []\n        for seed in seeds:\n            generator = random.Random(seed)\n            width = CONFIG["evaluation_joint_jitter_rad"]\n            offsets.append([generator.uniform(-width, width) for _ in range(qpos.shape[1] - 7)])\n        qpos[:, 7:] += torch.tensor(offsets, device=self.device, dtype=gs.tc_float)\n        self.evaluation_initial_qpos = qpos.detach().cpu().tolist()\n        self.robot.set_qpos(qpos, zero_velocity=True, skip_forward=False)\n        self.base_pos = self.robot.get_pos()\n        self.base_quat = self.robot.get_quat()\n        self.base_euler = quat_to_xyz(self.base_quat, rpy=True, degrees=True)\n        self.projected_gravity = transform_by_quat(self.global_gravity, inv_quat(self.base_quat))\n        self.dof_pos = self.robot.get_dofs_position(self.motors_dof_idx)\n        self.dof_vel.zero_()\n        self._update_observation()\n        return self.get_observations()\n', 'calibrate_push.py': '"""GPU physical calibration of the inspected Genesis 1.4.5 wrench API."""\nimport os\nos.environ[\'MPLBACKEND\'] = \'Agg\'\nimport argparse\nimport json\nfrom pathlib import Path\nimport torch\nimport genesis as gs\n\np = argparse.ArgumentParser()\np.add_argument(\'--output\', type=Path, required=True)\na = p.parse_args()\na.output.mkdir(parents=True, exist_ok=False)\ngs.init(backend=gs.cuda, precision=\'32\', logging_level=\'warning\', seed=81,\n        performance_mode=True)\nscene = gs.Scene(sim_options=gs.options.SimOptions(dt=.02, substeps=2, gravity=(0,0,0)),\n                 show_viewer=False)\nbody = scene.add_entity(gs.morphs.Box(size=(.2,.2,.2), pos=(0,0,2)),\n                        material=gs.materials.Rigid(rho=1000))\nscene.build(n_envs=2)\nmass = float(body.get_mass().flatten()[0])\nlink = body.links[0]\nforce = torch.tensor([[[0.,mass,0.]],[[0.,0.,0.]]], device=gs.device)\nrows=[]\nfor step in range(20):\n    # Ten 20ms steps, not twenty physics substeps; world-frame force at link COM.\n    if step < 10:\n        scene.rigid_solver.apply_links_external_wrench(force=force, links_idx=[link.idx],\n                                                       ref=gs.link_ref_frame.link_COM, local=False)\n    scene.step()\n    velocity=body.get_vel().cpu().tolist()\n    rows.append({\'step\':step,\'time_s\':(step+1)*.02,\'velocity\':velocity})\nexpected=.2\nactual=rows[9][\'velocity\'][0][1]\nassert abs(actual-expected)<.01,(actual,expected)\nassert max(abs(x) for r in rows for x in r[\'velocity\'][1])<1e-5,\'environment force leak\'\nassert abs(rows[-1][\'velocity\'][0][1]-actual)<1e-4,\'force persisted after release\'\n# Reset velocity and step without any wrench: no residual force may remain.\nbody.set_qpos(body.get_qpos(),zero_velocity=True,skip_forward=False)\nscene.step()\nassert float(body.get_vel().abs().max())<1e-5,\'force survived reset\'\nresult={\'status\':\'passed\',\'mass_kg\':mass,\'force_N\':mass,\'expected_delta_v\':expected,\n        \'observed_delta_v\':actual,\'dt\':.02,\'substeps\':2,\'pulse_steps\':10,\n        \'force_at\':\'link_COM\',\'local\':False,\'rows\':rows}\n(a.output/\'calibration.json\').write_text(json.dumps(result,indent=2))\nprint(json.dumps(result,indent=2))\n', 'command_runtime_checks.py': '"""Check the actual CUDA command sampler without building a physics scene."""\nimport os\nos.environ["MPLBACKEND"] = "Agg"\nimport torch\nfrom command_env import CommandEnv\nfrom commands import CONFIG\n\ntorch.manual_seed(13)\nenv = object.__new__(CommandEnv)\nenv.num_envs = 10000\nenv.device = torch.device("cuda")\nenv.evaluation = False\nenv.stage = CONFIG["stages"]["full"]\nenv.commands = torch.zeros(env.num_envs, 3, device=env.device)\nenv._resample_commands(None)\nc = env.commands\nassert torch.all(c[:, 1] == 0)\nassert torch.all((c[:, 0] >= 0) & (c[:, 0] <= 0.8))\nassert torch.all(c[:, 2].abs() <= 0.6)\nstop = (c == 0).all(dim=1)\nturn = (c[:, 0] == 0) & (c[:, 2] != 0)\nassert 0.17 < stop.float().mean().item() < 0.23\nassert 0.17 < turn.float().mean().item() < 0.23\nassert (c[:, 2] > 0).any() and (c[:, 2] < 0).any()\nmask = torch.arange(env.num_envs, device=env.device) % 2 == 0\nbefore = c.clone()\nenv._resample_commands(mask)\nassert torch.equal(env.commands[~mask], before[~mask])\nenv.stage = CONFIG["stages"]["speed"]\nenv._resample_commands(None)\nassert torch.all(env.commands[:, 2] == 0)\nassert torch.all(env.commands[:, 0] <= 0.6)\nprint("CUDA sampler: exact stops, both yaw signs, bounds and selective reset PASS")\n', 'audit_pushes.py': '"""Independent CSV audit: force timing, impulse and continuous recovery criteria."""\nimport argparse\nimport csv\nimport json\nimport math\nfrom pathlib import Path\n\n\ndef audit(run):\n    metrics=json.loads((run/\'metrics.json\').read_text())\n    mass=json.loads((run/\'force-api.json\').read_text())[\'mass_kg\']\n    groups={i:[] for i in range(len(metrics[\'trials\']))}\n    with (run/\'rollout.csv\').open() as f:\n        for raw in csv.DictReader(f):\n            row={k:float(v) for k,v in raw.items() if k not in (\'done\',\'termination\')}\n            row[\'done\']=raw[\'done\']==\'True\'\n            row[\'termination\']=raw[\'termination\']\n            assert all(math.isfinite(v) for v in row.values() if isinstance(v,(int,float)))\n            groups[int(row[\'trial\'])].append(row)\n    trials=[]\n    for i,t in enumerate(metrics[\'trials\']):\n        rs=groups[i]; start=t[\'push_start_step\']; end=start+10\n        assert all(int(r[\'step\'])==j and abs(r[\'time\']-(j+1)*.02)<1e-8 for j,r in enumerate(rs))\n        nonzero=[r for r in rs if math.hypot(r[\'fx\'],r[\'fy\'])>1e-5]\n        expected=[s for s in range(start,end) if s<len(rs)]\n        assert [int(r[\'step\']) for r in nonzero]==expected,\'wrong pulse steps\'\n        magnitude=t[\'alpha\']*mass*9.81\n        assert all(abs(math.hypot(r[\'fx\'],r[\'fy\'])-magnitude)<1e-4 and r[\'fz\']==0 for r in nonzero)\n        assert all(abs(r[k]-nonzero[0][k])<1e-5 for r in nonzero for k in (\'fx\',\'fy\',\'fz\'))\n        def good(r):return not r[\'done\'] and abs(r[\'vx\']-.5)<=.1 and abs(r[\'vy\'])<=.1 and abs(r[\'wz\'])<=.15\n        pre=rs[max(0,start-25):start]\n        pre_valid=len(pre)==25 and all(good(r) for r in pre)\n        dwell=0; recovered=None\n        for r in rs[end:]:\n            dwell=dwell+1 if good(r) else 0\n            if dwell==25 and recovered is None:recovered=(int(r[\'step\'])+1-end)*.02\n        success=len(rs)==600 and not any(r[\'done\'] for r in rs) and pre_valid and recovered is not None and recovered<=2+1e-8\n        assert success==t[\'success\'],(\'score mismatch\',i,success,t)\n        if recovered is not None:assert abs(recovered-t[\'recovery_s\'])<1e-6\n        trials.append(dict(trial=i,success=success,pre_valid=pre_valid,recovery_s=recovered,pulse_steps=len(nonzero),\n                           impulse_N_s=sum(math.hypot(r[\'fx\'],r[\'fy\'])*.02 for r in nonzero)))\n    return dict(status=\'passed\',trials=trials,successes=sum(t[\'success\'] for t in trials),count=len(trials))\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'run\',type=Path);a=p.parse_args()\n    result=audit(a.run)\n    (a.run/\'audit.json\').write_text(json.dumps(result,indent=2))\n    print(json.dumps(result,indent=2))\n', 'audit_terrain.py': '"""Recompute all terrain evaluation verdicts from before-reset CSV."""\nimport csv,json,math\nfrom pathlib import Path\ndef audit(run):\n    run=Path(run);m=json.loads((run/\'metrics.json\').read_text());initial=json.loads((run/\'initial-states.json\').read_text())\n    rows=list(csv.DictReader((run/\'rollout.csv\').open()));groups={i:[] for i in range(len(initial))}\n    for r in rows:groups[int(r[\'trial\'])].append(r)\n    verdicts=[]\n    for i,rs in groups.items():\n        assert rs and [int(r[\'step\']) for r in rs]==list(range(len(rs)))\n        assert all(abs(float(r[\'time\'])-(j+1)*.02)<1e-9 for j,r in enumerate(rs))\n        assert all(all(math.isfinite(float(v)) for k,v in r.items() if k not in [\'done\',\'solver_error\',\'outside_map\']) for r in rs)\n        assert all(float(r[\'cmd_vx\'])==(0 if j<50 else .5) for j,r in enumerate(rs))\n        assert all(r[\'done\']==\'False\' for r in rs[:-1]),\'CSV contains post-reset states\'\n        window=[r for r in rs if float(r[\'time\'])>2]\n        mae={k:sum(abs(float(r[k])-(.5 if k==\'vx\' else 0)) for r in window)/len(window) if window else None for k in [\'vx\',\'vy\',\'wz\']}\n        complete=len(rs)==600 and not any(r[k]==\'True\' for r in rs for k in [\'done\',\'solver_error\',\'outside_map\'])\n        distance=float(rs[-1][\'x\'])-initial[i][0];lateral=max(abs(float(r[\'y\'])-initial[i][1]) for r in rs)\n        success=complete and distance>4 and lateral<.75 and mae[\'vx\']<=.15 and mae[\'vy\']<=.15 and mae[\'wz\']<=.2\n        assert success==m[\'trials\'][i][\'success\'],(run,i,\'verdict mismatch\')\n        verdicts.append(success)\n    result=dict(status=\'passed\',trials=len(verdicts),successes=sum(verdicts),before_reset=True)\n    (run/\'audit.json\').write_text(json.dumps(result,indent=2));return result\nif __name__==\'__main__\':\n    import sys\n    print(json.dumps(audit(sys.argv[1]),indent=2))\n', 'record_terrain.py': '"""Prespecified seed200/A=.04, two 12s trials. Freeze frames after termination."""\nimport os\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport argparse,copy,csv,json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport genesis as gs\nimport av\nfrom PIL import Image,ImageDraw\nfrom terrain_maps import generate\nfrom terrain_env import TerrainEnv\nfrom rsl_rl.runners import OnPolicyRunner\np=argparse.ArgumentParser();p.add_argument(\'--checkpoint\',type=Path,required=True);p.add_argument(\'--run\',type=Path,required=True);p.add_argument(\'--label\',required=True);a=p.parse_args()\na.run.mkdir(parents=True,exist_ok=False)\nheight,meta=generate(200,.04);np.savez(a.run/\'terrain.npz\',height=height);(a.run/\'terrain.json\').write_text(json.dumps(meta,indent=2))\nsaved=json.loads((a.checkpoint.parent/\'config.json\').read_text());cfg=copy.deepcopy(saved[\'cfgs\']);cfg[0][\'episode_length_s\']=15.\ntorch.set_num_threads(2);gs.init(backend=gs.cuda,precision=\'32\',logging_level=\'warning\',seed=81,performance_mode=True)\nenv=TerrainEnv(1,cfg,a.run/\'terrain.npz\',evaluation=True,record=True)\nr=OnPolicyRunner(env,copy.deepcopy(saved[\'train_cfg\']),None,device=gs.device)\nr.load(str(a.checkpoint),load_cfg=dict(actor=True,critic=True,optimizer=False,iteration=False),map_location=str(gs.device));policy=r.get_inference_policy(device=gs.device)\nvideo=av.open(str(a.run/\'terrain.mp4\'),\'w\');stream=video.add_stream(\'libx264\',rate=25);stream.width,stream.height,stream.pix_fmt=640,480,\'yuv420p\'\nrows=[];initial=[];trials=[]\ntry:\n    for seed in [3000,3001]:\n        env.reset_with_yaws([0],[seed]);initial.append(env.evaluation_initial_qpos)\n        terminated=False;last_rgb=None;last_pos=env.robot.get_pos()[0].cpu().numpy();end_reason=None\n        env.camera.set_pose(pos=last_pos+np.array([1.8,-2.,1.25]),lookat=last_pos+np.array([0,0,-.1]));last_rgb=env.camera.render()[0]\n        with torch.no_grad():\n            for step in range(600):\n                command=[0,0,0] if step<50 else [.5,0,0]\n                if not terminated:\n                    obs=env.set_commands([command]);action=policy(obs);env.step(action);t=env.terminal\n                    pos=t[\'base_pos\'][0].cpu().numpy();vel=t[\'base_lin_vel\'][0].cpu().tolist();angles=t[\'base_euler\'][0].cpu().tolist()\n                    assert torch.isfinite(action).all() and np.isfinite(pos).all() and np.isfinite(vel).all()\n                    terminated=bool(t[\'reset_buf\'][0]) or bool(t[\'solver_error\'][0])\n                    if terminated:end_reason=\'solver_error\' if bool(t[\'solver_error\'][0]) else (\'outside_map\' if max(abs(pos[:2]))>11.5 else \'termination_roll_pitch_or_timeout\')\n                    row=dict(seed=seed,step=step,time=(step+1)*.02,cmd_vx=command[0],x=float(pos[0]),y=float(pos[1]),z=float(pos[2]),vx=vel[0],vy=vel[1],wz=float(t[\'base_ang_vel\'][0,2]),roll=angles[0],pitch=angles[1],done=terminated,ground_height=float(env.ground_height(t[\'base_pos\'][:,:2])[0]))\n                    rows.append(row)\n                    if not terminated:last_pos=pos.copy()\n                if step%2==0:\n                    if not terminated or last_rgb is None:\n                        env.camera.set_pose(pos=last_pos+np.array([1.8,-2.,1.25]),lookat=last_pos+np.array([0,0,-.1]));last_rgb=env.camera.render()[0]\n                    picture=Image.fromarray(last_rgb.copy());draw=ImageDraw.Draw(picture);draw.rectangle((0,0,640,76),fill=\'black\')\n                    draw.text((8,4),f\'{a.label} | seed200 A=+/-4cm | initial {seed} | playback 1x\',fill=\'white\')\n                    draw.text((8,23),f\'t={(step+1)*.02:.2f}s | command vx={command[0]:.2f} | actual vx={vel[0]:.2f}\',fill=\'white\')\n                    draw.text((8,43),f\'roll={angles[0]:.1f} pitch={angles[1]:.1f} deg | \'+(f\'TERMINATED: {end_reason}; FRAME FROZEN\' if terminated else \'running\'),fill=\'yellow\' if terminated else \'white\')\n                    for packet in stream.encode(av.VideoFrame.from_ndarray(np.asarray(picture),format=\'rgb24\')):video.mux(packet)\n        trials.append(dict(seed=seed,terminated=terminated,reason=end_reason))\nfinally:\n    for packet in stream.encode():video.mux(packet)\n    video.close()\n    if rows:\n        with (a.run/\'record.csv\').open(\'w\') as f:\n            w=csv.DictWriter(f,fieldnames=rows[0]);w.writeheader();w.writerows(rows)\n    (a.run/\'initial-states.json\').write_text(json.dumps(initial));(a.run/\'trials.json\').write_text(json.dumps(trials,indent=2))\nprint(\'Recorded two trials; terminated segments freeze the last pre-reset rendered frame.\')\n', 'commands.py': '"""Command schedules and trajectory scoring; usable without Torch/Genesis."""\nimport json\nimport math\nfrom pathlib import Path\n\nCONFIG = json.loads(Path(__file__).with_name("command-config.json").read_text())\nMODES = ("stop", "straight", "turn", "arc")\n\n\ndef schedule(stage="full", mirror=1):\n    if mirror not in (-1, 1):\n        raise ValueError("mirror must be +1 or -1")\n    end = [3, 7, 11, 15, 19, 23, 27, 30]\n    commands = [(0, 0, 0), (0.2, 0, 0), (0.7, 0, 0), (0, 0, 0),\n                (0, 0, 0.5 * mirror), (0, 0, -0.5 * mirror),\n                (0.4, 0, 0.4 * mirror), (0, 0, 0)]\n    if stage == "speed":\n        commands = [(0, 0, 0), (0.2, 0, 0), (0.5, 0, 0), (0, 0, 0),\n                    (0.3, 0, 0), (0, 0, 0), (0.5, 0, 0), (0, 0, 0)]\n    return [{"start": 0 if i == 0 else end[i-1], "end": end[i], "command": list(c)}\n            for i, c in enumerate(commands)]\n\n\ndef command_at(step, stage="full", mirror=1):\n    # Integer boundaries avoid drift from repeatedly adding floating point dt.\n    for index, segment in enumerate(schedule(stage, mirror)):\n        if step < round(segment["end"] / CONFIG["dt"]):\n            return index, segment["command"]\n    raise ValueError("step outside schedule")\n\n\ndef cases(formal=False, stage="full", recording=False):\n    if recording:\n        return [{"seed": 90, "yaw_degrees": 0, "mirror": 1}]\n    seeds = CONFIG["formal_seeds"] if formal else [90]\n    yaws = CONFIG["formal_yaw_degrees"] if formal else [0]\n    mirrors = CONFIG["mirrors"] if stage == "full" else [1]\n    return [{"seed": seed, "yaw_degrees": yaw, "mirror": mirror}\n            for yaw in yaws for mirror in mirrors for seed in seeds]\n\n\ndef score_trial(rows, stage="full", mirror=1, thresholds=None):\n    limits = thresholds or CONFIG["thresholds"]\n    dt = CONFIG["dt"]\n    segments = []\n    failure = any(r["done"] for r in rows)\n    for index, seg in enumerate(schedule(stage, mirror)):\n        group = [r for r in rows if r["segment"] == index]\n        cmd = seg["command"]\n        stop = cmd == [0, 0, 0]\n        turn = cmd[0] == 0 and cmd[2] != 0\n        stable = [r for r in group if r["time"] - seg["start"] > limits["exclude_s"] and not r["done"]]\n        def mean(fn):\n            return sum(fn(r) for r in stable) / len(stable) if stable else None\n        vx = mean(lambda r: abs(r["vx"] - cmd[0]))\n        vy = mean(lambda r: abs(r["vy"]))\n        yaw = mean(lambda r: abs(r["wz"] - cmd[2]))\n        xy = mean(lambda r: math.hypot(r["vx"], r["vy"]))\n        consecutive, settled = 0, None\n        for row in group:\n            if stop:\n                good = math.hypot(row["vx"], row["vy"]) < limits["stop_xy_speed"] and abs(row["wz"]) < limits["stop_yaw_speed"]\n            else:\n                good = abs(row["vx"]-cmd[0]) <= limits["vx_mae"] and abs(row["vy"]) <= limits["vy_mae"] and abs(row["wz"]-cmd[2]) <= limits["yaw_mae"]\n                if turn:\n                    good = good and math.hypot(row["vx"], row["vy"]) <= limits["turn_xy_speed"]\n            consecutive = consecutive + 1 if good and not row["done"] else 0\n            if settled is None and consecutive >= round(limits["dwell_s"] / dt):\n                settled = row["time"] - seg["start"]\n        complete = len(group) == round((seg["end"]-seg["start"]) / dt)\n        tracked = vx is not None and vx <= limits["vx_mae"] and vy <= limits["vy_mae"] and yaw <= limits["yaw_mae"]\n        if turn:\n            tracked = tracked and xy <= limits["turn_xy_speed"]\n        distance = sum(math.hypot(r["vx"], r["vy"]) * dt for r in group)\n        segments.append({"index": index, **seg, "steps": len(group), "vx_mae": vx, "vy_mae": vy,\n                         "yaw_mae": yaw, "xy_speed_mean": xy, "settled_at_s": settled,\n                         "travel_distance_m": distance,\n                         "success": bool(complete and tracked and settled is not None and settled <= limits["settle_s"] and not any(r["done"] for r in group))})\n    return {"steps": len(rows), "terminated_early": failure,\n            "success": len(rows) == 1500 and not failure and all(s["success"] for s in segments),\n            "max_abs_roll_degrees": max((abs(r["roll"]) for r in rows), default=None),\n            "max_abs_pitch_degrees": max((abs(r["pitch"]) for r in rows), default=None),\n            "segments": segments}\n'}
for name,text in FILES.items():(SOURCE/name).write_text(text)
def run(*args):
    subprocess.run([str(PY),*[str(x) for x in args]],cwd=SOURCE,env=ENV,check=True)
TAG=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"-"+uuid.uuid4().hex[:6]


## 外乱：校正と追加学習
外力校正→旧モデルの評価→上限付き学習・新旧比較。独立した新sessionに保存する。


In [ ]:
PUSH=BASE/"push-runs"/TAG
PUSH.mkdir(parents=True,exist_ok=False)
PB=Path("/content/drive/MyDrive/genesis-go2/push-runs")/TAG
run(SOURCE/"calibrate_push.py","--output",PUSH/"calibration")
run(SOURCE/"push_worker.py","eval","--run",PUSH/"dev-baseline","--checkpoint",CHECKPOINT)
run(SOURCE/"push_pipeline.py","--session",PUSH,"--checkpoint",CHECKPOINT,"--backup",PB)
run(SOURCE/"finish_push.py","--session",PUSH,"--backup",PB,"--old",CHECKPOINT)


## 不整地：高さ校正→短縮学習と再開→初回追加学習
01の公式ソースと02の指示環境から床を差し替える。既存ソースを残すため別フォルダで実行する。


In [ ]:
T=BASE/"terrain-runs"/TAG
T.mkdir(parents=True,exist_ok=False)
TB=Path("/content/drive/MyDrive/genesis-go2/terrain-runs")/TAG
TS=BASE/("terrain-source-"+TAG)
shutil.copytree(SOURCE,TS)
run(TS/"build_terrain_env.py","--source",SOURCE,"--output",TS)
run(TS/"calibrate_terrain_sampler.py","--output",T/"sampler-calibration.json")
run(TS/"terrain_preflight.py","--session",T,"--checkpoint",CHECKPOINT,"--calibration",T/"sampler-calibration.json","--backup",TB)
run(TS/"terrain_pipeline.py","--session",T,"--old",CHECKPOINT,"--backup",TB)
run(TS/"finish_terrain.py","--session",T,"--old",CHECKPOINT,"--backup",TB)


## 再挑戦（任意）
初回とは別のsource/sessionを作る。固定学習率と姿勢報酬の2候補を開発評価・平地回帰で選び、該当なしなら元モデルを維持する。


In [ ]:
RS=BASE/("terrain-retry-source-"+TAG)
run(SOURCE/"prepare_terrain_retry.py","--source",TS,"--output",RS)
R=BASE/"terrain-retry-runs"/TAG
RB=Path("/content/drive/MyDrive/genesis-go2/terrain-retry-runs")/TAG
run(RS/"terrain_retry_pipeline.py","--session",R,"--old",CHECKPOINT,"--backup",RB,"--calibration",T/"sampler-calibration.json")
run(RS/"finish_terrain.py","--session",R,"--old",CHECKPOINT,"--backup",RB)
